# Cross-Dataset Drift Analysis
## Análise de Drift entre Datasets JAFFE e CK+ para Reconhecimento de Expressões Faciais

**Metodologia:**
- SVM + LBP Multi-escala
- Balanceamento 50/50 com Data Augmentation  
- Validação: LOSO + K-Fold=3
- Análise de Cross-Dataset Drift

**Datasets:**
- JAFFE: Japanese Female Facial Expression Database
- CK+: Extended Cohn-Kanade Dataset

In [8]:
# ========================================
# IMPORTS E CONFIGURAÇÕES GLOBAIS
# ========================================

import os
import cv2
import numpy as np
import hashlib
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import random
import warnings
from scipy.stats import wasserstein_distance
import numpy as np
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy
from skimage.feature import local_binary_pattern


from scipy.stats import ks_2samp, entropy
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, 
                           accuracy_score, f1_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import shuffle
from sklearn.metrics.pairwise import cosine_similarity
from skimage.measure import shannon_entropy
from collections import defaultdict

# LBP
from skimage.feature import local_binary_pattern

# Configurações globais
CONFIG_FINAL = {
    'image_size': (96, 96),
    'lbp_radius': [1, 2, 3],
    'lbp_neighbors': [8, 16, 24],
    'lbp_method': 'uniform',
    'grid_size': (3, 3),
    'random_state': 42,
    'k_fold_cv': 3,
    'classes_alvo': ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sadness', 'surprise'],
    'balanceamento_ratio': 1.0
}

np.random.seed(CONFIG_FINAL['random_state'])
random.seed(CONFIG_FINAL['random_state'])

print("✅ Imports e configurações carregados!")
print("⚙️ Configurações:", CONFIG_FINAL)

✅ Imports e configurações carregados!
⚙️ Configurações: {'image_size': (96, 96), 'lbp_radius': [1, 2, 3], 'lbp_neighbors': [8, 16, 24], 'lbp_method': 'uniform', 'grid_size': (3, 3), 'random_state': 42, 'k_fold_cv': 3, 'classes_alvo': ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sadness', 'surprise'], 'balanceamento_ratio': 1.0}


In [9]:
# ========================================
# BALANCEADOR HÍBRIDO CROSS-DATASET COMPLETO
# ========================================

import os
import cv2
import numpy as np
import hashlib
import random
from collections import defaultdict, Counter
from scipy.stats import entropy
from sklearn.metrics.pairwise import cosine_similarity
from skimage.feature import local_binary_pattern
from tqdm import tqdm

def shannon_entropy(image):
    """Calcula entropia de Shannon para uma imagem."""
    histogram, _ = np.histogram(image, bins=256, range=(0, 256), density=True)
    histogram = histogram[histogram > 0]  # Remove zeros para evitar log(0)
    return entropy(histogram, base=2)

def redimensionar_preservando_aspecto(imagem, target_size=(128, 128)):
    """
    Redimensiona imagem preservando proporções e usando padding.
    Minimiza perda de informação facial comparado ao redimensionamento direto.
    
    Args:
        imagem: Imagem de entrada (numpy array)
        target_size: Tamanho alvo (width, height)
    
    Returns:
        Imagem redimensionada com padding preservando aspecto
    """
    h, w = imagem.shape
    aspect_ratio = w / h
    
    if aspect_ratio > 1:  # Mais largo que alto
        new_w = target_size[0]
        new_h = int(target_size[0] / aspect_ratio)
    else:  # Mais alto que largo
        new_h = target_size[1]
        new_w = int(target_size[1] * aspect_ratio)
    
    # Redimensionar mantendo proporção
    resized = cv2.resize(imagem, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
    
    # Calcular padding para centralizar
    pad_h = (target_size[1] - new_h) // 2
    pad_w = (target_size[0] - new_w) // 2
    
    # Criar imagem final com padding
    result = np.zeros(target_size, dtype=imagem.dtype)
    result[pad_h:pad_h+new_h, pad_w:pad_w+new_w] = resized
    
    return result

def preprocessar_para_otimizar_resolucao(imagem):
    """
    Aplica pré-processamento para preservar informação em resolução reduzida.
    Baseado em técnicas de literatura para FER em baixa resolução.
    
    Args:
        imagem: Imagem normalizada (0-1) ou uint8
    
    Returns:
        Imagem pré-processada otimizada para baixa resolução
    """
    # Converter para uint8 se necessário
    if imagem.dtype != np.uint8:
        if imagem.max() <= 1.0:
            img_uint8 = (imagem * 255).astype(np.uint8)
        else:
            img_uint8 = imagem.astype(np.uint8)
    else:
        img_uint8 = imagem.copy()
    
    # 1. Equalização de histograma adaptativa (CLAHE) para realçar contraste local
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_eq = clahe.apply(img_uint8)
    
    # 2. Filtro bilateral para preservar bordas enquanto reduz ruído
    img_filtered = cv2.bilateralFilter(img_eq, 9, 75, 75)
    
    # 3. Sharpening sutil para realçar detalhes faciais
    kernel_sharpen = np.array([[-0.5,-0.5,-0.5], 
                              [-0.5, 5.0,-0.5], 
                              [-0.5,-0.5,-0.5]])
    img_sharp = cv2.filter2D(img_filtered, -1, kernel_sharpen)
    
    # Normalizar para 0-1
    return np.clip(img_sharp / 255.0, 0, 1)

class BalanceadorHibrido:
    """
    Implementa estratégia híbrida de balanceamento cross-dataset
    com distribuição inteligente, mínima repetição de imagens base
    e redimensionamento preservando aspecto.
    """
    
    def __init__(self, random_state=42, target_size=(128, 128), usar_preprocessamento=True):
        self.random_state = random_state
        self.target_size = target_size
        self.usar_preprocessamento = usar_preprocessamento
        
        np.random.seed(random_state)
        random.seed(random_state)
        
        self.scores_qualidade = {}
        self.scores_unicidade = {}
        self.quotas_distribuidas = {}
        self.transformacoes_por_imagem = defaultdict(set)
        
        print(f"🔧 BalanceadorHibrido configurado:")
        print(f"   • Resolução alvo: {target_size}")
        print(f"   • Pré-processamento: {'Ativado' if usar_preprocessamento else 'Desativado'}")
        print(f"   • Preservação de aspecto: Ativada")
        
    def processar_imagem_entrada(self, imagem):
        """
        Processa imagem de entrada com redimensionamento inteligente.
        
        Args:
            imagem: Imagem original
            
        Returns:
            Imagem processada e otimizada
        """
        # Redimensionar preservando aspecto
        img_resized = redimensionar_preservando_aspecto(imagem, self.target_size)
        
        # Aplicar pré-processamento se ativado
        if self.usar_preprocessamento:
            img_processed = preprocessar_para_otimizar_resolucao(img_resized)
        else:
            # Normalizar para 0-1 se necessário
            if img_resized.dtype != np.float32:
                if img_resized.max() > 1.0:
                    img_processed = img_resized.astype(np.float32) / 255.0
                else:
                    img_processed = img_resized.astype(np.float32)
            else:
                img_processed = img_resized
        
        return img_processed
        
    def executar_balanceamento_cross_dataset(self, dados_jaffe, dados_ck, classes_alvo):
        """
        Executa balanceamento híbrido cross-dataset com processamento inteligente.
        Iguala JAFFE ao CK+ para cada classe usando estratégia inteligente.
        """
        
        print("🧠 EXECUTANDO BALANCEAMENTO HÍBRIDO CROSS-DATASET")
        print("=" * 70)
        print("• Análise de qualidade e unicidade das imagens")
        print("• Distribuição inteligente de quotas de repetição")
        print("• Geração de sintéticas com máxima diversidade")
        print("• Processamento inteligente de resolução")
        print("• Objetivo: Igualar JAFFE ao CK+ (expansão, não redução)")
        
        # Processar imagens de entrada com redimensionamento inteligente
        print(f"\n🔄 Processando imagens com resolução {self.target_size}...")
        
        dados_jaffe_processados = []
        for img, cls, suj in dados_jaffe:
            img_processada = self.processar_imagem_entrada(img)
            dados_jaffe_processados.append((img_processada, cls, suj))
        
        dados_ck_processados = []
        for img, cls, suj in dados_ck:
            img_processada = self.processar_imagem_entrada(img)
            dados_ck_processados.append((img_processada, cls, suj))
        
        print(f"✅ Processamento concluído:")
        print(f"   • JAFFE: {len(dados_jaffe_processados)} imagens processadas")
        print(f"   • CK+: {len(dados_ck_processados)} imagens processadas")
        
        # Filtrar dados
        dados_jaffe_filt = [(img, cls, suj) for img, cls, suj in dados_jaffe_processados if cls in classes_alvo]
        dados_ck_filt = [(img, cls, suj) for img, cls, suj in dados_ck_processados if cls in classes_alvo]
        
        # Agrupar por classe
        dados_por_classe_jaffe = self._agrupar_por_classe(dados_jaffe_filt)
        dados_por_classe_ck = self._agrupar_por_classe(dados_ck_filt)
        
        # Executar balanceamento por classe
        dados_jaffe_result = []
        dados_ck_result = []
        stats_completos = {}
        
        print(f"\n📊 PLANO DE BALANCEAMENTO POR CLASSE:")
        print("Classe       JAFFE    CK+     Target   Sintéticas_J  Sintéticas_C")
        print("-" * 75)
        
        for classe in classes_alvo:
            dados_jaffe_classe = dados_por_classe_jaffe.get(classe, [])
            dados_ck_classe = dados_por_classe_ck.get(classe, [])
            
            if not dados_jaffe_classe or not dados_ck_classe:
                print(f"{classe:12} {len(dados_jaffe_classe):6d}   {len(dados_ck_classe):6d}   SKIP")
                continue
            
            # Target: igualar ao maior (estratégia de expansão)
            target = max(len(dados_jaffe_classe), len(dados_ck_classe))
            sinteticas_jaffe = max(0, target - len(dados_jaffe_classe))
            sinteticas_ck = max(0, target - len(dados_ck_classe))
            
            print(f"{classe:12} {len(dados_jaffe_classe):6d}   {len(dados_ck_classe):6d}   {target:6d}   {sinteticas_jaffe:11d}   {sinteticas_ck:11d}")
            
            # Processar JAFFE
            jaffe_processado = self._processar_dataset_classe(
                dados_jaffe_classe, dados_ck_classe, target, "JAFFE", classe
            )
            
            # Processar CK+
            ck_processado = self._processar_dataset_classe(
                dados_ck_classe, dados_jaffe_classe, target, "CK+", classe
            )
            
            dados_jaffe_result.extend(jaffe_processado)
            dados_ck_result.extend(ck_processado)
            
            stats_completos[classe] = {
                'jaffe_original': len(dados_jaffe_classe),
                'ck_original': len(dados_ck_classe),
                'target': target,
                'jaffe_sinteticas': sinteticas_jaffe,
                'ck_sinteticas': sinteticas_ck,
                'jaffe_final': len(jaffe_processado),
                'ck_final': len(ck_processado)
            }
        
        print("-" * 75)
        print(f"{'TOTAL':12} {len(dados_jaffe_result):6d}   {len(dados_ck_result):6d}")
        
        # Gerar relatório final
        self._gerar_relatorio_hibrido(stats_completos)
        
        return dados_jaffe_result, dados_ck_result, stats_completos
    
    def _agrupar_por_classe(self, dados):
        """Agrupa dados por classe."""
        grupos = defaultdict(list)
        for img, cls, suj in dados:
            grupos[cls].append((img, cls, suj))
        return dict(grupos)
    
    def _processar_dataset_classe(self, dados_classe, dados_referencia, target, dataset_name, classe):
        """
        Processa uma classe específica aplicando estratégia híbrida.
        """
        
        dados_expandido = dados_classe.copy()  # Adicionar todas as originais
        sinteticas_needed = max(0, target - len(dados_classe))
        
        if sinteticas_needed == 0:
            return dados_expandido
        
        # Análise de qualidade e unicidade (apenas para JAFFE, CK+ usa augmentation simples)
        if dataset_name == "JAFFE" and sinteticas_needed > 0:
            print(f"\n🔍 Análise híbrida para {classe} (JAFFE)...")
            
            # Análise de qualidade
            scores_qualidade = self._avaliar_qualidade_imagens(dados_classe)
            
            # Análise de unicidade vs dataset de referência
            scores_unicidade = self._calcular_unicidade_vs_referencia(dados_classe, dados_referencia)
            
            # Distribuição inteligente de quotas
            quotas = self._distribuir_quotas_inteligente(
                dados_classe, scores_qualidade, scores_unicidade, sinteticas_needed
            )
            
            # Geração de sintéticas com máxima diversidade
            sinteticas_geradas = self._gerar_sinteticas_hibrido(dados_classe, quotas, classe)
            
        else:
            # CK+: augmentation simples
            sinteticas_geradas = []
            for i in range(sinteticas_needed):
                img_base, cls, suj_base = random.choice(dados_classe)
                tecnica = np.random.choice(['horizontal_flip', 'brightness_up', 'brightness_down', 'rotation_left'])
                img_sintetica = self._aplicar_transformacao_especifica(img_base, tecnica)
                suj_sintetico = f"{suj_base}_{dataset_name.lower()}_aug_{i:03d}"
                sinteticas_geradas.append((img_sintetica, cls, suj_sintetico))
        
        dados_expandido.extend(sinteticas_geradas)
        return dados_expandido
    
    def _avaliar_qualidade_imagens(self, dados_classe):
        """
        Avalia qualidade das imagens baseado em nitidez, contraste e entropia.
        """
        
        scores = {}
        
        for img, cls, suj in dados_classe:
            # Converter para uint8 se necessário
            if img.dtype != np.uint8:
                if img.max() <= 1.0:
                    img_uint8 = (img * 255).astype(np.uint8)
                else:
                    img_uint8 = img.astype(np.uint8)
            else:
                img_uint8 = img
            
            # Métricas de qualidade
            # 1. Nitidez (Laplacian variance)
            nitidez = cv2.Laplacian(img_uint8, cv2.CV_64F).var()
            
            # 2. Contraste (desvio padrão)
            contraste = np.std(img_uint8)
            
            # 3. Entropia (complexidade da informação)
            entropia = shannon_entropy(img_uint8)
            
            # 4. Detecção de bordas (Canny)
            edges = cv2.Canny(img_uint8, 50, 150)
            densidade_bordas = np.sum(edges > 0) / edges.size
            
            # Score composto (normalizado 0-10)
            score_nitidez = min(nitidez / 500, 10)
            score_contraste = min(contraste / 50, 10)
            score_entropia = min(entropia / 8, 10)
            score_bordas = densidade_bordas * 100
            
            score_final = (score_nitidez + score_contraste + score_entropia + score_bordas) / 4
            scores[suj] = min(score_final, 10.0)
        
        return scores
    
    def _calcular_unicidade_vs_referencia(self, dados_classe, dados_referencia):
        """
        Calcula unicidade das imagens comparado ao dataset de referência usando LBP.
        """
        
        scores_unicidade = {}
        
        # Extrair features LBP simples para comparação
        features_classe = []
        sujeitos_classe = []
        
        for img, cls, suj in dados_classe:
            features = self._extrair_features_lbp_simples(img)
            features_classe.append(features)
            sujeitos_classe.append(suj)
        
        features_referencia = []
        for img, cls, suj in dados_referencia:
            features = self._extrair_features_lbp_simples(img)
            features_referencia.append(features)
        
        features_classe = np.array(features_classe)
        features_referencia = np.array(features_referencia)
        
        # Calcular similaridade
        for i, suj in enumerate(sujeitos_classe):
            # Similaridade cosseno com todas as imagens de referência
            similaridades = cosine_similarity([features_classe[i]], features_referencia)[0]
            
            # Score de unicidade = 1 - max_similaridade
            max_similaridade = np.max(similaridades)
            score_unicidade = (1 - max_similaridade) * 10
            
            scores_unicidade[suj] = max(score_unicidade, 0.0)
        
        return scores_unicidade
    
    def _extrair_features_lbp_simples(self, img):
        """Extrai features LBP simples para comparação."""
        if img.dtype != np.uint8:
            if img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            else:
                img = img.astype(np.uint8)
        
        # LBP simples
        lbp = local_binary_pattern(img, 8, 1, method='uniform')
        hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)
        return hist
    
    def _distribuir_quotas_inteligente(self, dados_classe, scores_qualidade, scores_unicidade, total_needed):
        """
        Distribui quotas de repetição baseado em qualidade e unicidade.
        """
        
        # Combinar scores (60% qualidade, 40% unicidade)
        scores_finais = {}
        for suj in scores_qualidade.keys():
            score_qualidade = scores_qualidade[suj]
            score_unicidade = scores_unicidade[suj]
            score_final = (score_qualidade * 0.6) + (score_unicidade * 0.4)
            scores_finais[suj] = score_final
        
        # Ordenar por score (melhores primeiro)
        sujeitos_ordenados = sorted(scores_finais.keys(), key=lambda x: scores_finais[x], reverse=True)
        
        # Distribuição baseada em tiers
        n_imagens = len(dados_classe)
        quota_base = total_needed // n_imagens
        quota_extra = total_needed % n_imagens
        
        quotas = {}
        tier_size = max(1, n_imagens // 3)
        
        for i, suj in enumerate(sujeitos_ordenados):
            if i < tier_size:  # Tier A (melhor terço)
                quota = quota_base + 2
            elif i < 2 * tier_size:  # Tier B (terço médio)
                quota = quota_base + 1
            else:  # Tier C (terço inferior)
                quota = quota_base
            
            quotas[suj] = max(quota, 0)
        
        # Distribuir quotas extras para os melhores
        extras_distribuidas = 0
        for suj in sujeitos_ordenados:
            if extras_distribuidas < quota_extra:
                quotas[suj] += 1
                extras_distribuidas += 1
            else:
                break
        
        # Ajuste fino para atingir total exato
        total_quotas = sum(quotas.values())
        if total_quotas != total_needed:
            diff = total_needed - total_quotas
            if diff > 0:
                for suj in sujeitos_ordenados[:diff]:
                    quotas[suj] += 1
            else:
                for suj in reversed(sujeitos_ordenados[:abs(diff)]):
                    quotas[suj] = max(0, quotas[suj] - 1)
        
        return quotas
    
    def _gerar_sinteticas_hibrido(self, dados_classe, quotas, classe):
        """
        Gera sintéticas com máxima diversidade usando quotas inteligentes.
        """
        
        sinteticas = []
        
        # Criar mapeamento sujeito -> dados
        mapa_sujeito = {}
        for img, cls, suj in dados_classe:
            mapa_sujeito[suj] = (img, cls, suj)
        
        # Gerar sintéticas por quota
        for suj, quota in quotas.items():
            if quota <= 0:
                continue
                
            img_base, cls, _ = mapa_sujeito[suj]
            
            # Gerar transformações únicas para esta imagem
            sinteticas_geradas = self._gerar_transformacoes_unicas(
                img_base, cls, suj, quota
            )
            
            sinteticas.extend(sinteticas_geradas)
        
        return sinteticas
    
    def _gerar_transformacoes_unicas(self, img_base, classe, sujeito_base, num_sinteticas):
        """
        Gera transformações únicas garantindo máxima diversidade.
        """
        
        sinteticas = []
        transformacoes_usadas = set()
        
        # Definir todas as combinações possíveis
        tecnicas_basicas = [
            'horizontal_flip', 'rotation_left', 'rotation_right',
            'brightness_up', 'brightness_down', 'contrast_up', 'contrast_down',
            'translation', 'noise'
        ]
        
        for i in range(num_sinteticas):
            transformacao_unica = None
            tentativas = 0
            max_tentativas = 50
            
            while tentativas < max_tentativas:
                # Escolher técnica(s)
                if np.random.random() < 0.7:  # 70% uma técnica
                    tecnicas = [np.random.choice(tecnicas_basicas)]
                else:  # 30% duas técnicas
                    tecnicas = np.random.choice(tecnicas_basicas, 2, replace=False).tolist()
                
                # Criar hash da combinação
                combinacao_hash = self._hash_combinacao(tecnicas)
                
                # Verificar se já foi usada para esta imagem
                key_imagem = f"{sujeito_base}_{combinacao_hash}"
                if key_imagem not in transformacoes_usadas:
                    transformacoes_usadas.add(key_imagem)
                    transformacao_unica = tecnicas
                    break
                
                tentativas += 1
            
            if transformacao_unica is None:
                # Fallback: usar técnica aleatória
                transformacao_unica = [np.random.choice(tecnicas_basicas)]
            
            # Aplicar transformação
            img_sintetica = img_base.copy()
            for tecnica in transformacao_unica:
                img_sintetica = self._aplicar_transformacao_especifica(img_sintetica, tecnica)
            
            # Validar qualidade
            if self._validar_qualidade_sintetica(img_sintetica, img_base):
                sujeito_sintetico = f"{sujeito_base}_hyb_{i:03d}"
                sinteticas.append((img_sintetica, classe, sujeito_sintetico))
            else:
                # Fallback seguro
                img_sintetica = self._aplicar_transformacao_especifica(img_base, 'horizontal_flip')
                sujeito_sintetico = f"{sujeito_base}_hyb_{i:03d}_safe"
                sinteticas.append((img_sintetica, classe, sujeito_sintetico))
        
        return sinteticas
    
    def _hash_combinacao(self, tecnicas):
        """Cria hash único para combinação de técnicas."""
        tecnicas_str = "_".join(sorted(tecnicas))
        return hashlib.md5(tecnicas_str.encode()).hexdigest()[:8]
    
    def _aplicar_transformacao_especifica(self, imagem, tecnica):
        """Aplica transformação específica mantendo qualidade facial."""
        if imagem.dtype != np.float32:
            imagem = imagem.astype(np.float32)
        
        h, w = imagem.shape
        center = (w // 2, h // 2)
        
        if tecnica == 'horizontal_flip':
            return cv2.flip(imagem, 1)
        
        elif tecnica == 'rotation_left':
            angle = np.random.uniform(-8, -3)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            return cv2.warpAffine(imagem, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        
        elif tecnica == 'rotation_right':
            angle = np.random.uniform(3, 8)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            return cv2.warpAffine(imagem, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        
        elif tecnica == 'brightness_up':
            factor = np.random.uniform(1.05, 1.25)
            return np.clip(imagem * factor, 0, 1)
        
        elif tecnica == 'brightness_down':
            factor = np.random.uniform(0.75, 0.95)
            return np.clip(imagem * factor, 0, 1)
        
        elif tecnica == 'contrast_up':
            alpha = np.random.uniform(1.1, 1.3)
            return np.clip(alpha * (imagem - 0.5) + 0.5, 0, 1)
        
        elif tecnica == 'contrast_down':
            alpha = np.random.uniform(0.7, 0.9)
            return np.clip(alpha * (imagem - 0.5) + 0.5, 0, 1)
        
        elif tecnica == 'translation':
            tx = np.random.randint(-5, 6)
            ty = np.random.randint(-5, 6)
            M = np.float32([[1, 0, tx], [0, 1, ty]])
            return cv2.warpAffine(imagem, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        
        elif tecnica == 'noise':
            noise = np.random.normal(0, 0.015, imagem.shape)
            return np.clip(imagem + noise, 0, 1)
        
        else:
            return imagem  # Fallback
    
    def _validar_qualidade_sintetica(self, img_sintetica, img_original):
        """Valida se a imagem sintética mantém qualidade adequada."""
        
        # Verificações básicas
        if np.any(np.isnan(img_sintetica)) or np.any(np.isinf(img_sintetica)):
            return False
        
        if img_sintetica.std() < 0.01:  # Muito uniforme
            return False
        
        # Verificar correlação com original
        try:
            correlation = np.corrcoef(img_original.flatten(), img_sintetica.flatten())[0, 1]
            if correlation < 0.2:  # Muito diferente
                return False
        except:
            pass
        
        return True
    
    def _gerar_relatorio_hibrido(self, stats_completos):
        """Gera relatório detalhado da estratégia híbrida."""
        
        print("\n" + "=" * 70)
        print("📊 RELATÓRIO FINAL - ESTRATÉGIA HÍBRIDA")
        print("=" * 70)
        
        # Resumo geral
        total_jaffe_orig = sum([s['jaffe_original'] for s in stats_completos.values()])
        total_ck_orig = sum([s['ck_original'] for s in stats_completos.values()])
        total_jaffe_final = sum([s['jaffe_final'] for s in stats_completos.values()])
        total_ck_final = sum([s['ck_final'] for s in stats_completos.values()])
        total_sinteticas = sum([s['jaffe_sinteticas'] + s['ck_sinteticas'] for s in stats_completos.values()])
        
        print(f"\n📈 RESUMO GERAL:")
        print(f"   • JAFFE: {total_jaffe_orig} → {total_jaffe_final} amostras")
        print(f"   • CK+: {total_ck_orig} → {total_ck_final} amostras")
        print(f"   • Total sintéticas geradas: {total_sinteticas}")
        print(f"   • Ganho total: {(total_jaffe_final + total_ck_final) - (total_jaffe_orig + total_ck_orig)}")
        print(f"   • Resolução processada: {self.target_size}")
        print(f"   • Pré-processamento aplicado: {'Sim' if self.usar_preprocessamento else 'Não'}")
        
        # Ratio final
        if total_jaffe_final + total_ck_final > 0:
            ratio_final = (total_ck_final / (total_jaffe_final + total_ck_final)) * 100
            print(f"\n🎯 EFICIÊNCIA DA ESTRATÉGIA:")
            print(f"   • Ratio final: {100-ratio_final:.1f}% JAFFE / {ratio_final:.1f}% CK+")
            print(f"   • Desvio do 50/50: {abs(50-ratio_final):.1f} pontos percentuais")
            
            if abs(50 - ratio_final) < 3:
                print("   🎉 EXCELENTE: Balanceamento quase perfeito!")
            elif abs(50 - ratio_final) < 7:
                print("   ✅ BOM: Balanceamento adequado.")
            else:
                print("   ⚠️ ACEITÁVEL: Pequeno desvio do ideal.")


# ========================================
# FUNÇÃO PRINCIPAL PARA USAR NO NOTEBOOK
# ========================================

def executar_balanceamento_hibrido_com_resolucao_otimizada(dados_jaffe, dados_ck, 
                                                          classes_alvo=None, 
                                                          target_size=(128, 128), 
                                                          usar_preprocessamento=True,
                                                          random_state=42):
    """
    Função principal para executar balanceamento híbrido com redimensionamento inteligente.
    
    Esta função substitui completamente a executar_balanceamento_50_50() e incorpora
    as melhores práticas de redimensionamento baseadas em literatura científica.
    
    Args:
        dados_jaffe: Lista de tuplas (imagem, classe, sujeito) do JAFFE
        dados_ck: Lista de tuplas (imagem, classe, sujeito) do CK+
        classes_alvo: Lista de classes para processar (None = todas disponíveis)
        target_size: Resolução alvo (width, height). Recomendado: (128, 128) ou (160, 160)
        usar_preprocessamento: Se True, aplica técnicas de otimização para baixa resolução
        random_state: Seed para reprodutibilidade
    
    Returns:
        dados_jaffe_balanced: JAFFE balanceado com estratégia híbrida e resolução otimizada
        dados_ck_balanced: CK+ balanceado (se necessário)
        stats_balanceamento: Estatísticas detalhadas do balanceamento
    """
    
    # Definir classes padrão se não especificadas
    if classes_alvo is None:
        # Extrair classes comuns entre os datasets
        classes_jaffe = set([cls for _, cls, _ in dados_jaffe])
        classes_ck = set([cls for _, cls, _ in dados_ck])
        classes_alvo = sorted(list(classes_jaffe.intersection(classes_ck)))
        print(f"🎯 Classes detectadas automaticamente: {classes_alvo}")
    
    # Inicializar balanceador com configurações otimizadas
    balanceador = BalanceadorHibrido(
        random_state=random_state,
        target_size=target_size,
        usar_preprocessamento=usar_preprocessamento
    )
    
    print(f"\n🚀 Iniciando processamento com configurações:")
    print(f"   • Classes alvo: {len(classes_alvo)} classes")
    print(f"   • Resolução: {target_size}")
    print(f"   • Pré-processamento: {'Ativado' if usar_preprocessamento else 'Desativado'}")
    
    # Executar balanceamento
    dados_jaffe_balanced, dados_ck_balanced, stats = balanceador.executar_balanceamento_cross_dataset(
        dados_jaffe, dados_ck, classes_alvo
    )
    
    print(f"\n🎉 BALANCEAMENTO HÍBRIDO COM RESOLUÇÃO OTIMIZADA CONCLUÍDO!")
    print(f"   • JAFFE balanceado: {len(dados_jaffe_balanced)} amostras")
    print(f"   • CK+ balanceado: {len(dados_ck_balanced)} amostras")
    print(f"   • Resolução final: {target_size}")
    print(f"   • Preservação de aspecto: Aplicada")
    
    return dados_jaffe_balanced, dados_ck_balanced, stats


# ========================================
# FUNÇÃO DE TESTE E VALIDAÇÃO
# ========================================

def validar_qualidade_processamento(dados_originais, dados_processados, num_amostras=5):
    """
    Valida a qualidade do processamento de redimensionamento comparando
    imagens originais com as processadas.
    
    Args:
        dados_originais: Lista de dados antes do processamento
        dados_processados: Lista de dados após processamento
        num_amostras: Número de amostras aleatórias para validar
    
    Returns:
        dict: Estatísticas de qualidade do processamento
    """
    
    print("🔍 VALIDAÇÃO DA QUALIDADE DO PROCESSAMENTO")
    print("=" * 50)
    
    import random
    
    # Selecionar amostras aleatórias
    indices = random.sample(range(min(len(dados_originais), len(dados_processados))), 
                           min(num_amostras, len(dados_originais)))
    
    stats_qualidade = {
        'correlacoes': [],
        'ssim_scores': [],
        'entropia_original': [],
        'entropia_processada': [],
        'nitidez_original': [],
        'nitidez_processada': []
    }
    
    for i, idx in enumerate(indices):
        img_orig, cls, suj = dados_originais[idx]
        img_proc, _, _ = dados_processados[idx]
        
        # Redimensionar original para comparação justa
        if img_orig.shape != img_proc.shape:
            img_orig_resized = cv2.resize(img_orig, img_proc.shape[::-1])
        else:
            img_orig_resized = img_orig
        
        # Normalizar para comparação
        if img_orig_resized.max() > 1.0:
            img_orig_resized = img_orig_resized / 255.0
        if img_proc.max() > 1.0:
            img_proc = img_proc / 255.0
            
        # Calcular métricas
        # 1. Correlação
        correlation = np.corrcoef(img_orig_resized.flatten(), img_proc.flatten())[0, 1]
        stats_qualidade['correlacoes'].append(correlation)
        
        # 2. Entropia
        img_orig_uint8 = (img_orig_resized * 255).astype(np.uint8)
        img_proc_uint8 = (img_proc * 255).astype(np.uint8)
        
        entropia_orig = shannon_entropy(img_orig_uint8)
        entropia_proc = shannon_entropy(img_proc_uint8)
        
        stats_qualidade['entropia_original'].append(entropia_orig)
        stats_qualidade['entropia_processada'].append(entropia_proc)
        
        # 3. Nitidez (Laplacian)
        nitidez_orig = cv2.Laplacian(img_orig_uint8, cv2.CV_64F).var()
        nitidez_proc = cv2.Laplacian(img_proc_uint8, cv2.CV_64F).var()
        
        stats_qualidade['nitidez_original'].append(nitidez_orig)
        stats_qualidade['nitidez_processada'].append(nitidez_proc)
        
        print(f"Amostra {i+1} ({cls}-{suj}):")
        print(f"   • Correlação: {correlation:.3f}")
        print(f"   • Entropia: {entropia_orig:.2f} → {entropia_proc:.2f}")
        print(f"   • Nitidez: {nitidez_orig:.1f} → {nitidez_proc:.1f}")
    
    # Calcular médias
    print(f"\n📊 RESUMO DA QUALIDADE:")
    print(f"   • Correlação média: {np.mean(stats_qualidade['correlacoes']):.3f}")
    print(f"   • Entropia preservada: {np.mean(stats_qualidade['entropia_processada'])/np.mean(stats_qualidade['entropia_original'])*100:.1f}%")
    print(f"   • Nitidez preservada: {np.mean(stats_qualidade['nitidez_processada'])/np.mean(stats_qualidade['nitidez_original'])*100:.1f}%")
    
    # Interpretação dos resultados
    correlacao_media = np.mean(stats_qualidade['correlacoes'])
    if correlacao_media > 0.8:
        print("   ✅ EXCELENTE: Alta preservação de informação")
    elif correlacao_media > 0.6:
        print("   ✅ BOM: Boa preservação de informação")
    elif correlacao_media > 0.4:
        print("   ⚠️ ADEQUADO: Preservação moderada")
    else:
        print("   ❌ ATENÇÃO: Baixa preservação - considere ajustar parâmetros")
    
    return stats_qualidade


# ========================================
# EXEMPLO DE USO COMPLETO
# ========================================

def exemplo_uso_completo():
    """
    Exemplo completo de como usar o sistema de balanceamento híbrido
    com redimensionamento inteligente.
    """
    
    print("📋 EXEMPLO DE USO - BALANCEAMENTO HÍBRIDO COM RESOLUÇÃO OTIMIZADA")
    print("=" * 70)
    
    # Exemplo de uso básico
    print("\n1️⃣ USO BÁSICO (Configuração recomendada):")
    print("""
# Carregar dados (substitua pela sua função de carregamento)
# dados_jaffe = carregar_imagens_por_sujeito('/path/to/jaffe')
# dados_ck = carregar_imagens_por_sujeito('/path/to/ck+')

# Executar balanceamento com configuração otimizada
dados_jaffe_balanced, dados_ck_balanced, stats = executar_balanceamento_hibrido_com_resolucao_otimizada(
    dados_jaffe, 
    dados_ck,
    target_size=(128, 128),  # Resolução otimizada baseada em literatura
    usar_preprocessamento=True  # Técnicas de otimização para baixa resolução
)
""")
    
    print("\n2️⃣ USO AVANÇADO (Configuração customizada):")
    print("""
# Configuração para experimentos específicos
balanceador = BalanceadorHibrido(
    target_size=(160, 160),  # Resolução maior se computacionalmente viável
    usar_preprocessamento=True,
    random_state=42
)

# Execução com classes específicas
classes_emocionais = ['anger', 'happiness', 'sadness', 'surprise', 'fear', 'disgust', 'neutral']
dados_jaffe_balanced, dados_ck_balanced, stats = balanceador.executar_balanceamento_cross_dataset(
    dados_jaffe, dados_ck, classes_emocionais
)

# Validar qualidade do processamento
stats_qualidade = validar_qualidade_processamento(dados_jaffe, dados_jaffe_balanced)
""")
    
    print("\n3️⃣ CONFIGURAÇÕES RECOMENDADAS POR CONTEXTO:")
    print("""
# Para desenvolvimento/prototipagem rápida:
target_size=(96, 96), usar_preprocessamento=False

# Para pesquisa acadêmica (recomendado):
target_size=(128, 128), usar_preprocessamento=True

# Para máxima qualidade (se recursos permitirem):
target_size=(160, 160), usar_preprocessamento=True

# Para sistemas embarcados/tempo real:
target_size=(64, 64), usar_preprocessamento=True
""")


# ========================================
# FUNÇÕES AUXILIARES EXTRAS
# ========================================

def gerar_relatorio_detalhado(stats_balanceamento):
    """
    Gera relatório detalhado das estatísticas de balanceamento.
    
    Args:
        stats_balanceamento: Dicionário com estatísticas do balanceamento
    """
    print("\n📊 RELATÓRIO DETALHADO DE BALANCEAMENTO")
    print("=" * 60)
    
    for classe, stats in stats_balanceamento.items():
        print(f"\n🎭 CLASSE: {classe.upper()}")
        print(f"   📈 Original JAFFE: {stats['jaffe_original']}")
        print(f"   📈 Original CK+: {stats['ck_original']}")
        print(f"   🎯 Target: {stats['target']}")
        print(f"   🔧 Sintéticas JAFFE: {stats['jaffe_sinteticas']}")
        print(f"   🔧 Sintéticas CK+: {stats['ck_sinteticas']}")
        print(f"   ✅ Final JAFFE: {stats['jaffe_final']}")
        print(f"   ✅ Final CK+: {stats['ck_final']}")
        
        # Calcular eficiência
        if stats['jaffe_original'] > 0:
            aumento_jaffe = ((stats['jaffe_final'] - stats['jaffe_original']) / stats['jaffe_original']) * 100
            print(f"   📊 Aumento JAFFE: +{aumento_jaffe:.1f}%")


def verificar_qualidade_imagens(dados, num_amostras=10):
    """
    Verifica a qualidade geral das imagens processadas.
    
    Args:
        dados: Lista de tuplas (imagem, classe, sujeito)
        num_amostras: Número de amostras para análise
    
    Returns:
        dict: Estatísticas de qualidade
    """
    print(f"🔍 ANÁLISE DE QUALIDADE - {num_amostras} amostras")
    print("-" * 40)
    
    indices = np.random.choice(len(dados), min(num_amostras, len(dados)), replace=False)
    
    nitidez_scores = []
    contraste_scores = []
    entropia_scores = []
    
    for idx in indices:
        img, cls, suj = dados[idx]
        
        # Converter para uint8 se necessário
        if img.dtype != np.uint8:
            if img.max() <= 1.0:
                img_uint8 = (img * 255).astype(np.uint8)
            else:
                img_uint8 = img.astype(np.uint8)
        else:
            img_uint8 = img
        
        # Calcular métricas
        nitidez = cv2.Laplacian(img_uint8, cv2.CV_64F).var()
        contraste = np.std(img_uint8)
        entropia = shannon_entropy(img_uint8)
        
        nitidez_scores.append(nitidez)
        contraste_scores.append(contraste)
        entropia_scores.append(entropia)
    
    stats = {
        'nitidez_media': np.mean(nitidez_scores),
        'contraste_medio': np.mean(contraste_scores),
        'entropia_media': np.mean(entropia_scores),
        'nitidez_std': np.std(nitidez_scores),
        'contraste_std': np.std(contraste_scores),
        'entropia_std': np.std(entropia_scores)
    }
    
    print(f"📊 Nitidez: {stats['nitidez_media']:.1f} ± {stats['nitidez_std']:.1f}")
    print(f"📊 Contraste: {stats['contraste_medio']:.1f} ± {stats['contraste_std']:.1f}")
    print(f"📊 Entropia: {stats['entropia_media']:.2f} ± {stats['entropia_std']:.2f}")
    
    return stats


print("✅ Balanceador Híbrido Cross-Dataset Completo implementado!")
print("🎯 Principais características:")
print("   • Preservação de aspecto das imagens")
print("   • Pré-processamento para otimização de resolução")
print("   • Análise de qualidade e unicidade")
print("   • Distribuição inteligente de quotas")
print("   • 9 transformações diferentes para data augmentation")
print("   • Validação de qualidade integrada")
print("   • Relatórios detalhados")
print("\n🚀 Use: executar_balanceamento_hibrido_com_resolucao_otimizada(dados_jaffe, dados_ck)")
print("📖 Para exemplos: exemplo_uso_completo()")

✅ Balanceador Híbrido Cross-Dataset Completo implementado!
🎯 Principais características:
   • Preservação de aspecto das imagens
   • Pré-processamento para otimização de resolução
   • Análise de qualidade e unicidade
   • Distribuição inteligente de quotas
   • 9 transformações diferentes para data augmentation
   • Validação de qualidade integrada
   • Relatórios detalhados

🚀 Use: executar_balanceamento_hibrido_com_resolucao_otimizada(dados_jaffe, dados_ck)
📖 Para exemplos: exemplo_uso_completo()


In [ ]:
print("   ⚠️ ACEITÁVEL: Pequeno desvio do ideal.")


# ========================================
# FUNÇÃO PRINCIPAL PARA USAR NO NOTEBOOK
# ========================================

def executar_balanceamento_hibrido_com_resolucao_otimizada(dados_jaffe, dados_ck, 
                                                          classes_alvo=None, 
                                                          target_size=(128, 128), 
                                                          usar_preprocessamento=True,
                                                          random_state=42):
    """
    Função principal para executar balanceamento híbrido com redimensionamento inteligente.
    
    Esta função substitui completamente a executar_balanceamento_50_50() e incorpora
    as melhores práticas de redimensionamento baseadas em literatura científica.
    
    Args:
        dados_jaffe: Lista de tuplas (imagem, classe, sujeito) do JAFFE
        dados_ck: Lista de tuplas (imagem, classe, sujeito) do CK+
        classes_alvo: Lista de classes para processar (None = todas disponíveis)
        target_size: Resolução alvo (width, height). Recomendado: (128, 128) ou (160, 160)
        usar_preprocessamento: Se True, aplica técnicas de otimização para baixa resolução
        random_state: Seed para reprodutibilidade
    
    Returns:
        dados_jaffe_balanced: JAFFE balanceado com estratégia híbrida e resolução otimizada
        dados_ck_balanced: CK+ balanceado (se necessário)
        stats_balanceamento: Estatísticas detalhadas do balanceamento
    """
    
    # Definir classes padrão se não especificadas
    if classes_alvo is None:
        # Extrair classes comuns entre os datasets
        classes_jaffe = set([cls for _, cls, _ in dados_jaffe])
        classes_ck = set([cls for _, cls, _ in dados_ck])
        classes_alvo = sorted(list(classes_jaffe.intersection(classes_ck)))
        print(f"🎯 Classes detectadas automaticamente: {classes_alvo}")
    
    # Inicializar balanceador com configurações otimizadas
    balanceador = BalanceadorHibrido(
        random_state=random_state,
        target_size=target_size,
        usar_preprocessamento=usar_preprocessamento
    )
    
    print(f"\n🚀 Iniciando processamento com configurações:")
    print(f"   • Classes alvo: {len(classes_alvo)} classes")
    print(f"   • Resolução: {target_size}")
    print(f"   • Pré-processamento: {'Ativado' if usar_preprocessamento else 'Desativado'}")
    
    # Executar balanceamento
    dados_jaffe_balanced, dados_ck_balanced, stats = balanceador.executar_balanceamento_cross_dataset(
        dados_jaffe, dados_ck, classes_alvo
    )
    
    print(f"\n🎉 BALANCEAMENTO HÍBRIDO COM RESOLUÇÃO OTIMIZADA CONCLUÍDO!")
    print(f"   • JAFFE balanceado: {len(dados_jaffe_balanced)} amostras")
    print(f"   • CK+ balanceado: {len(dados_ck_balanced)} amostras")
    print(f"   • Resolução final: {target_size}")
    print(f"   • Preservação de aspecto: Aplicada")
    
    return dados_jaffe_balanced, dados_ck_balanced, stats


# ========================================
# FUNÇÃO DE TESTE E VALIDAÇÃO
# ========================================

def validar_qualidade_processamento(dados_originais, dados_processados, num_amostras=5):
    """
    Valida a qualidade do processamento de redimensionamento comparando
    imagens originais com as processadas.
    
    Args:
        dados_originais: Lista de dados antes do processamento
        dados_processados: Lista de dados após processamento
        num_amostras: Número de amostras aleatórias para validar
    
    Returns:
        dict: Estatísticas de qualidade do processamento
    """
    
    print("🔍 VALIDAÇÃO DA QUALIDADE DO PROCESSAMENTO")
    print("=" * 50)
    
    import random
    
    # Selecionar amostras aleatórias
    indices = random.sample(range(min(len(dados_originais), len(dados_processados))), 
                           min(num_amostras, len(dados_originais)))
    
    stats_qualidade = {
        'correlacoes': [],
        'ssim_scores': [],
        'entropia_original': [],
        'entropia_processada': [],
        'nitidez_original': [],
        'nitidez_processada': []
    }
    
    for i, idx in enumerate(indices):
        img_orig, cls, suj = dados_originais[idx]
        img_proc, _, _ = dados_processados[idx]
        
        # Redimensionar original para comparação justa
        if img_orig.shape != img_proc.shape:
            img_orig_resized = cv2.resize(img_orig, img_proc.shape[::-1])
        else:
            img_orig_resized = img_orig
        
        # Normalizar para comparação
        if img_orig_resized.max() > 1.0:
            img_orig_resized = img_orig_resized / 255.0
        if img_proc.max() > 1.0:
            img_proc = img_proc / 255.0
            
        # Calcular métricas
        # 1. Correlação
        correlation = np.corrcoef(img_orig_resized.flatten(), img_proc.flatten())[0, 1]
        stats_qualidade['correlacoes'].append(correlation)
        
        # 2. Entropia
        img_orig_uint8 = (img_orig_resized * 255).astype(np.uint8)
        img_proc_uint8 = (img_proc * 255).astype(np.uint8)
        
        entropia_orig = shannon_entropy(img_orig_uint8)
        entropia_proc = shannon_entropy(img_proc_uint8)
        
        stats_qualidade['entropia_original'].append(entropia_orig)
        stats_qualidade['entropia_processada'].append(entropia_proc)
        
        # 3. Nitidez (Laplacian)
        nitidez_orig = cv2.Laplacian(img_orig_uint8, cv2.CV_64F).var()
        nitidez_proc = cv2.Laplacian(img_proc_uint8, cv2.CV_64F).var()
        
        stats_qualidade['nitidez_original'].append(nitidez_orig)
        stats_qualidade['nitidez_processada'].append(nitidez_proc)
        
        print(f"Amostra {i+1} ({cls}-{suj}):")
        print(f"   • Correlação: {correlation:.3f}")
        print(f"   • Entropia: {entropia_orig:.2f} → {entropia_proc:.2f}")
        print(f"   • Nitidez: {nitidez_orig:.1f} → {nitidez_proc:.1f}")
    
    # Calcular médias
    print(f"\n📊 RESUMO DA QUALIDADE:")
    print(f"   • Correlação média: {np.mean(stats_qualidade['correlacoes']):.3f}")
    print(f"   • Entropia preservada: {np.mean(stats_qualidade['entropia_processada'])/np.mean(stats_qualidade['entropia_original'])*100:.1f}%")
    print(f"   • Nitidez preservada: {np.mean(stats_qualidade['nitidez_processada'])/np.mean(stats_qualidade['nitidez_original'])*100:.1f}%")
    
    # Interpretação dos resultados
    correlacao_media = np.mean(stats_qualidade['correlacoes'])
    if correlacao_media > 0.8:
        print("   ✅ EXCELENTE: Alta preservação de informação")
    elif correlacao_media > 0.6:
        print("   ✅ BOM: Boa preservação de informação")
    elif correlacao_media > 0.4:
        print("   ⚠️ ADEQUADO: Preservação moderada")
    else:
        print("   ❌ ATENÇÃO: Baixa preservação - considere ajustar parâmetros")
    
    return stats_qualidade


# ========================================
# EXEMPLO DE USO COMPLETO
# ========================================

def exemplo_uso_completo():
    """
    Exemplo completo de como usar o sistema de balanceamento híbrido
    com redimensionamento inteligente.
    """
    
    print("📋 EXEMPLO DE USO - BALANCEAMENTO HÍBRIDO COM RESOLUÇÃO OTIMIZADA")
    print("=" * 70)
    
    # Exemplo de uso básico
    print("\n1️⃣ USO BÁSICO (Configuração recomendada):")
    print("""
# Carregar dados (substitua pela sua função de carregamento)
# dados_jaffe = carregar_imagens_por_sujeito('/path/to/jaffe')
# dados_ck = carregar_imagens_por_sujeito('/path/to/ck+')

# Executar balanceamento com configuração otimizada
dados_jaffe_balanced, dados_ck_balanced, stats = executar_balanceamento_hibrido_com_resolucao_otimizada(
    dados_jaffe, 
    dados_ck,
    target_size=(128, 128),  # Resolução otimizada baseada em literatura
    usar_preprocessamento=True  # Técnicas de otimização para baixa resolução
)
""")
    
    print("\n2️⃣ USO AVANÇADO (Configuração customizada):")
    print("""
# Configuração para experimentos específicos
balanceador = BalanceadorHibrido(
    target_size=(160, 160),  # Resolução maior se computacionalmente viável
    usar_preprocessamento=True,
    random_state=42
)

# Execução com classes específicas
classes_emocionais = ['anger', 'happiness', 'sadness', 'surprise', 'fear', 'disgust', 'neutral']
dados_jaffe_balanced, dados_ck_balanced, stats = balanceador.executar_balanceamento_cross_dataset(
    dados_jaffe, dados_ck, classes_emocionais
)

# Validar qualidade do processamento
stats_qualidade = validar_qualidade_processamento(dados_jaffe, dados_jaffe_balanced)
""")
    
    print("\n3️⃣ CONFIGURAÇÕES RECOMENDADAS POR CONTEXTO:")
    print("""
# Para desenvolvimento/prototipagem rápida:
target_size=(96, 96), usar_preprocessamento=False

# Para pesquisa acadêmica (recomendado):
target_size=(128, 128), usar_preprocessamento=True

# Para máxima qualidade (se recursos permitirem):
target_size=(160, 160), usar_preprocessamento=True

# Para sistemas embarcados/tempo real:
target_size=(64, 64), usar_preprocessamento=True
""")


print("✅ Balanceador Híbrido com Redimensionamento Inteligente implementado!")
print("🎯 Principais melhorias:")
print("   • Preservação de aspecto das imagens")
print("   • Pré-processamento para otimização de resolução")
print("   • Configuração flexível de resolução alvo")
print("   • Validação de qualidade integrada")
print("   • Baseado em literatura científica de FER")
print("\n🚀 Use: executar_balanceamento_hibrido_com_resolucao_otimizada(dados_jaffe, dados_ck)")

IndentationError: unexpected indent (1013711817.py, line 1)

In [ ]:
# ========================================
# CLASSIFICADOR SVM
# ========================================

class SVMClassifier:
    """Classificador SVM com otimização de hiperparâmetros."""
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.best_model = None
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        self.best_params = None
        self.grid_search = None
        
    def definir_grid_parametros(self, tipo_grid='medio'):
        """Define grid de parâmetros para otimização."""
        if tipo_grid == 'rapido':
            param_grid = [
                {'kernel': ['linear'], 'C': [0.1, 1, 10]},
                {'kernel': ['rbf'], 'C': [1, 10], 'gamma': ['scale', 0.001]}
            ]
        elif tipo_grid == 'medio':
            param_grid = [
                {'kernel': ['linear'], 'C': [0.01, 0.1, 1, 10, 100]},
                {'kernel': ['rbf'], 'C': [0.1, 1, 10, 100], 
                 'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]}
            ]
        else:  # completo
            param_grid = [
                {'kernel': ['linear'], 'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]},
                {'kernel': ['rbf'], 'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
                 'gamma': ['scale', 'auto', 0.0001, 0.001, 0.01, 0.1, 1]}
            ]
        return param_grid
    
    def treinar(self, X_train, y_train, tipo_grid='medio', cv_folds=3, 
                scoring='f1_macro', n_jobs=-1, verbose=True):
        """Treina SVM com grid search."""
        if verbose:
            print(f"🚀 Treinando SVM: {X_train.shape[0]} amostras, {X_train.shape[1]} características")
        
        if len(X_train) < cv_folds:
            cv_folds = max(2, len(X_train) - 1)
        
        X_train_scaled = self.scaler.fit_transform(X_train)
        y_train_encoded = self.label_encoder.fit_transform(y_train)
        
        param_grid = self.definir_grid_parametros(tipo_grid)
        svm_base = SVC(random_state=self.random_state, probability=True)
        
        try:
            self.grid_search = GridSearchCV(
                estimator=svm_base,
                param_grid=param_grid,
                cv=StratifiedKFold(n_splits=cv_folds, shuffle=True, 
                                 random_state=self.random_state),
                scoring=scoring,
                n_jobs=n_jobs,
                verbose=1 if verbose else 0,
                return_train_score=True
            )
        except Exception as e:
            print(f"⚠️ Erro no GridSearch: {e}")
            self.grid_search = GridSearchCV(
                estimator=svm_base,
                param_grid=[{'kernel': ['linear'], 'C': [1]}],
                cv=2,
                scoring=scoring
            )
        
        self.grid_search.fit(X_train_scaled, y_train_encoded)
        self.best_model = self.grid_search.best_estimator_
        self.best_params = self.grid_search.best_params_
        
        if verbose:
            print(f"✅ Melhor score: {self.grid_search.best_score_:.4f}")
            print(f"   Parâmetros: {self.best_params}")
        
        return self
    
    def predizer(self, X_test):
        """Faz predições."""
        if self.best_model is None:
            raise ValueError("Modelo não treinado!")
        
        X_test_scaled = self.scaler.transform(X_test)
        y_pred_encoded = self.best_model.predict(X_test_scaled)
        return self.label_encoder.inverse_transform(y_pred_encoded)
    
    def avaliar(self, X_test, y_test, verbose=True):
        """Avalia modelo."""
        y_pred = self.predizer(X_test)
        
        accuracy = accuracy_score(y_test, y_pred)
        
        try:
            f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
            f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
            f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        except:
            f1_macro = f1_micro = f1_weighted = 0.0
        
        try:
            class_report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
            conf_matrix = confusion_matrix(y_test, y_pred)
        except:
            class_report = {}
            conf_matrix = np.array([])
        
        metricas = {
            'accuracy': accuracy,
            'f1_macro': f1_macro,
            'f1_micro': f1_micro,
            'f1_weighted': f1_weighted,
            'classification_report': class_report,
            'confusion_matrix': conf_matrix,
            'y_true': y_test,
            'y_pred': y_pred
        }
        
        if verbose:
            print(f"📊 Accuracy: {accuracy:.4f}, F1-macro: {f1_macro:.4f}")
        
        return metricas

In [ ]:

# ========================================
# ANÁLISE DE CROSS-DATASET DRIFT
# ========================================

class CrossDatasetDriftAnalyzer:
    """Analisador completo de cross-dataset drift."""
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.datasets = {}
        self.drift_metrics = {}
        self.cross_performance = {}
        self.loso_results = {}
        
    def adicionar_dataset(self, nome, X, y, subjects=None):
        """Adiciona dataset para análise."""
        self.datasets[nome] = {
            'X': X,
            'y': y,
            'subjects': subjects if subjects is not None else np.arange(len(X)),
            'n_samples': len(X),
            'n_features': X.shape[1],
            'classes': np.unique(y),
            'class_distribution': Counter(y)
        }
        
        print(f"✅ Dataset '{nome}': {len(X)} amostras, {X.shape[1]} características")
        print(f"   Classes: {list(np.unique(y))}")
        print(f"   Sujeitos únicos: {len(np.unique(self.datasets[nome]['subjects']))}")
    
    def calcular_drift_estatistico(self, dataset1, dataset2, n_features_sample=100):
        """Calcula métricas estatísticas de drift."""
        print(f"🔍 Analisando drift: {dataset1} vs {dataset2}")
        
        X1 = self.datasets[dataset1]['X']
        X2 = self.datasets[dataset2]['X']
        
        if X1.shape[1] > n_features_sample:
            feature_indices = np.random.RandomState(self.random_state).choice(
                X1.shape[1], n_features_sample, replace=False
            )
            X1_sample = X1[:, feature_indices]
            X2_sample = X2[:, feature_indices]
        else:
            X1_sample = X1
            X2_sample = X2
        
        drift_metrics = {}
        
        # Kolmogorov-Smirnov Test
        ks_pvalues = []
        ks_statistics = []
        
        for i in range(X1_sample.shape[1]):
            ks_stat, p_value = ks_2samp(X1_sample[:, i], X2_sample[:, i])
            ks_statistics.append(ks_stat)
            ks_pvalues.append(p_value)
        
        drift_metrics['ks_test'] = {
            'mean_statistic': np.mean(ks_statistics),
            'mean_pvalue': np.mean(ks_pvalues),
            'drift_percentage': np.mean(np.array(ks_pvalues) < 0.05) * 100
        }
        
        # Wasserstein Distance
        wasserstein_distances = []
        for i in range(min(50, X1_sample.shape[1])):
            try:
                wd = wasserstein_distance(X1_sample[:, i], X2_sample[:, i])
                wasserstein_distances.append(wd)
            except:
                continue
        
        drift_metrics['wasserstein'] = {
            'mean_distance': np.mean(wasserstein_distances),
            'std_distance': np.std(wasserstein_distances)
        }
        
        # Distribuição de classes
        dist1 = self.datasets[dataset1]['class_distribution']
        dist2 = self.datasets[dataset2]['class_distribution']
        
        classes_comuns = set(dist1.keys()) & set(dist2.keys())
        if classes_comuns:
            prob1 = np.array([dist1.get(c, 0) for c in classes_comuns])
            prob2 = np.array([dist2.get(c, 0) for c in classes_comuns])
            
            prob1 = prob1 / prob1.sum()
            prob2 = prob2 / prob2.sum()
            
            kl_div = entropy(prob1, prob2)
            
            drift_metrics['class_distribution'] = {
                'kl_divergence': kl_div,
                'common_classes': list(classes_comuns),
                'dist1': dict(zip(classes_comuns, prob1)),
                'dist2': dict(zip(classes_comuns, prob2))
            }
        
        self.drift_metrics[f"{dataset1}_vs_{dataset2}"] = drift_metrics
        
        print(f"   📈 Drift: {drift_metrics['ks_test']['drift_percentage']:.1f}% características")
        print(f"   📏 Wasserstein: {drift_metrics['wasserstein']['mean_distance']:.4f}")
        
        return drift_metrics
    
    def avaliar_cross_performance(self, source_dataset, target_dataset, verbose=True):
        """Avalia performance cross-dataset."""
        print(f"🔄 Cross-performance: {source_dataset} → {target_dataset}")
        
        X_train = self.datasets[source_dataset]['X']
        y_train = self.datasets[source_dataset]['y']
        X_test = self.datasets[target_dataset]['X']
        y_test = self.datasets[target_dataset]['y']
        
        # Filtrar classes comuns
        classes_comuns = set(y_train) & set(y_test)
        
        if not classes_comuns:
            print("   ⚠️ Nenhuma classe comum!")
            return None
        
        mask_train = np.isin(y_train, list(classes_comuns))
        mask_test = np.isin(y_test, list(classes_comuns))
        
        X_train_filt = X_train[mask_train]
        y_train_filt = y_train[mask_train]
        X_test_filt = X_test[mask_test]
        y_test_filt = y_test[mask_test]
        
        print(f"   🎭 Classes: {sorted(classes_comuns)}")
        print(f"   📊 Treino: {len(X_train_filt)}, Teste: {len(X_test_filt)}")
        
        resultados = {}
        
        # SVM
        try:
            svm = SVMClassifier()
            svm.treinar(X_train_filt, y_train_filt, tipo_grid='medio', verbose=False)
            metricas = svm.avaliar(X_test_filt, y_test_filt, verbose=False)
            
            resultados['SVM'] = metricas
            
            if verbose:
                print(f"   🤖 SVM: {metricas['accuracy']:.4f} acc, {metricas['f1_macro']:.4f} f1")
                
        except Exception as e:
            print(f"   ❌ Erro SVM: {e}")
        
        key = f"{source_dataset}_to_{target_dataset}"
        self.cross_performance[key] = resultados
        
        return resultados
    
    def executar_loso_interno(self, dataset_name, verbose=True):
        """Executa LOSO no dataset interno."""
        print(f"🎯 LOSO interno: {dataset_name}")
        
        X = self.datasets[dataset_name]['X']
        y = self.datasets[dataset_name]['y'] 
        subjects = self.datasets[dataset_name]['subjects']
        
        sujeitos_unicos = np.unique(subjects)
        resultados = []
        
        for i, sujeito_teste in enumerate(sujeitos_unicos):
            if verbose and i < 5:  # Mostra apenas primeiros 5 para não poluir
                print(f"   Fold {i+1}/{len(sujeitos_unicos)}: {sujeito_teste}")
            
            mask_teste = subjects == sujeito_teste
            mask_treino = ~mask_teste
            
            X_train = X[mask_treino]
            y_train = y[mask_treino]
            X_test = X[mask_teste]
            y_test = y[mask_teste]
            
            if len(X_test) == 0 or len(X_train) < 5:
                continue
                
            try:
                svm = SVMClassifier()
                svm.treinar(X_train, y_train, tipo_grid='rapido', verbose=False)
                metricas = svm.avaliar(X_test, y_test, verbose=False)
                
                resultados.append({
                    'sujeito': sujeito_teste,
                    'accuracy': metricas['accuracy'],
                    'f1_macro': metricas['f1_macro']
                })
                
            except Exception as e:
                if verbose:
                    print(f"      ❌ Erro: {e}")
                continue
        
        if resultados:
            accuracies = [r['accuracy'] for r in resultados]
            f1_scores = [r['f1_macro'] for r in resultados]
            
            stats = {
                'n_folds': len(resultados),
                'accuracy_mean': np.mean(accuracies),
                'accuracy_std': np.std(accuracies),
                'f1_mean': np.mean(f1_scores),
                'f1_std': np.std(f1_scores),
                'resultados': resultados
            }
            
            self.loso_results[dataset_name] = stats
            
            print(f"   📊 LOSO ({len(resultados)} folds): "
                  f"{stats['accuracy_mean']:.4f}±{stats['accuracy_std']:.4f}")
            
            return stats
        
        print("   ❌ Nenhum fold executado com sucesso")
        return None
    
    def executar_kfold_interno(self, dataset_name, k=3, verbose=True):
        """Executa K-Fold interno no dataset."""
        print(f"📊 K-Fold={k} interno: {dataset_name}")
        
        X = self.datasets[dataset_name]['X']
        y = self.datasets[dataset_name]['y']
        
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=self.random_state)
        accuracies = []
        f1_scores = []
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            try:
                svm = SVMClassifier()
                svm.treinar(X_train, y_train, tipo_grid='rapido', verbose=False)
                metricas = svm.avaliar(X_val, y_val, verbose=False)
                
                accuracies.append(metricas['accuracy'])
                f1_scores.append(metricas['f1_macro'])
                
                if verbose:
                    print(f"   Fold {fold+1}: {metricas['accuracy']:.4f}")
                    
            except Exception as e:
                if verbose:
                    print(f"   Fold {fold+1}: Erro - {e}")
                continue
        
        if accuracies:
            stats = {
                'k': k,
                'accuracy_mean': np.mean(accuracies),
                'accuracy_std': np.std(accuracies),
                'f1_mean': np.mean(f1_scores),
                'f1_std': np.std(f1_scores),
                'accuracies': accuracies,
                'f1_scores': f1_scores
            }
            
            print(f"   📊 K-Fold: {stats['accuracy_mean']:.4f}±{stats['accuracy_std']:.4f}")
            return stats
        
        print("   ❌ Nenhum fold executado")
        return None

In [ ]:
# ========================================
# FUNÇÕES DE VISUALIZAÇÃO
# ========================================

def gerar_visualizacoes_completas(resultados):
    """Gera visualizações completas dos resultados."""
    
    print("📊 Gerando visualizações...")
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Distribuição de classes após balanceamento
    ax1 = axes[0, 0]
    plot_distribuicao_classes(resultados, ax1)
    
    # 2. Performance cross-dataset
    ax2 = axes[0, 1] 
    plot_cross_performance(resultados, ax2)
    
    # 3. Comparação LOSO vs K-Fold
    ax3 = axes[0, 2]
    plot_validacao_interna(resultados, ax3)
    
    # 4. Drift metrics heatmap
    ax4 = axes[1, 0]
    plot_drift_metrics(resultados, ax4)
    
    # 5. PCA dos datasets
    ax5 = axes[1, 1]
    plot_pca_datasets(resultados, ax5)
    
    # 6. Resumo de performance
    ax6 = axes[1, 2]
    plot_resumo_performance(resultados, ax6)
    
    plt.suptitle('Cross-Dataset Drift Analysis - Resultados Completos', fontsize=16)
    plt.tight_layout()
    plt.show()

def plot_distribuicao_classes(resultados, ax):
    """Plota distribuição de classes."""
    jaffe_data = resultados['dados_processados']['jaffe']
    ck_data = resultados['dados_processados']['ck']
    
    jaffe_dist = Counter(jaffe_data[1])
    ck_dist = Counter(ck_data[1])
    
    classes = sorted(set(jaffe_dist.keys()) | set(ck_dist.keys()))
    
    jaffe_counts = [jaffe_dist.get(c, 0) for c in classes]
    ck_counts = [ck_dist.get(c, 0) for c in classes]
    
    x = np.arange(len(classes))
    width = 0.35
    
    ax.bar(x - width/2, jaffe_counts, width, label='JAFFE (balanceado)', alpha=0.8)
    ax.bar(x + width/2, ck_counts, width, label='CK+ (original)', alpha=0.8)
    
    ax.set_xlabel('Classes')
    ax.set_ylabel('Número de Amostras')
    ax.set_title('Distribuição de Classes')
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3)

def plot_cross_performance(resultados, ax):
    """Plota performance cross-dataset."""
    cross_perf = resultados['cross_performance']
    
    directions = []
    accuracies = []
    
    for direction, results in cross_perf.items():
        if results and 'SVM' in results:
            directions.append(direction.replace('_', '→').replace('to', ''))
            accuracies.append(results['SVM']['accuracy'])
    
    if directions:
        bars = ax.bar(directions, accuracies, color=['skyblue', 'lightcoral'])
        ax.set_ylabel('Accuracy')
        ax.set_title('Performance Cross-Dataset')
        ax.set_ylim(0, 1)
        
        # Adicionar valores nas barras
        for bar, acc in zip(bars, accuracies):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{acc:.3f}', ha='center', va='bottom')
    
    ax.grid(True, alpha=0.3)

def plot_validacao_interna(resultados, ax):
    """Plota comparação de validação interna."""
    validacao = resultados['validacao_interna']
    
    datasets = []
    loso_accs = []
    kfold_accs = []
    
    if validacao['loso_jaffe']:
        datasets.append('JAFFE')
        loso_accs.append(validacao['loso_jaffe']['accuracy_mean'])
        
    if validacao['kfold_jaffe']:
        if 'JAFFE' not in datasets:
            datasets.append('JAFFE')
        kfold_accs.append(validacao['kfold_jaffe']['accuracy_mean'])
    
    if validacao['loso_ck']:
        datasets.append('CK+')
        loso_accs.append(validacao['loso_ck']['accuracy_mean'])
        
    if validacao['kfold_ck']:
        if 'CK+' not in datasets:
            datasets.append('CK+')
        kfold_accs.append(validacao['kfold_ck']['accuracy_mean'])
    
    if datasets:
        x = np.arange(len(datasets))
        width = 0.35
        
        if loso_accs:
            ax.bar(x - width/2, loso_accs, width, label='LOSO', alpha=0.8)
        if kfold_accs:
            ax.bar(x + width/2, kfold_accs, width, label='K-Fold=3', alpha=0.8)
        
        ax.set_xlabel('Dataset')
        ax.set_ylabel('Accuracy')
        ax.set_title('Validação Interna')
        ax.set_xticks(x)
        ax.set_xticklabels(datasets)
        ax.legend()
        ax.grid(True, alpha=0.3)

def plot_drift_metrics(resultados, ax):
    """Plota métricas de drift."""
    drift = resultados['drift_metrics']
    
    if drift:
        metrics = ['KS Drift %', 'Wasserstein', 'KL Divergence']
        values = [
            drift['ks_test']['drift_percentage'],
            drift['wasserstein']['mean_distance'] * 100,  # Escalar para visualização
            drift.get('class_distribution', {}).get('kl_divergence', 0) * 10  # Escalar
        ]
        
        bars = ax.bar(metrics, values, color=['red', 'orange', 'yellow'], alpha=0.7)
        ax.set_ylabel('Magnitude')
        ax.set_title('Métricas de Drift')
        
        # Adicionar valores
        for bar, val in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + max(values)*0.01,
                   f'{val:.1f}', ha='center', va='bottom')
    
    ax.grid(True, alpha=0.3)

def plot_pca_datasets(resultados, ax):
    """Plota PCA dos datasets."""
    from sklearn.decomposition import PCA
    
    jaffe_data = resultados['dados_processados']['jaffe']
    ck_data = resultados['dados_processados']['ck']
    
    # Combinar dados para PCA consistente
    X_combined = np.vstack([jaffe_data[0], ck_data[0]])
    
    # PCA
    pca = PCA(n_components=2, random_state=CONFIG_FINAL['random_state'])
    X_pca = pca.fit_transform(X_combined)
    
    # Separar de volta
    n_jaffe = len(jaffe_data[0])
    X_jaffe_pca = X_pca[:n_jaffe]
    X_ck_pca = X_pca[n_jaffe:]
    
    ax.scatter(X_jaffe_pca[:, 0], X_jaffe_pca[:, 1], 
              c='blue', alpha=0.6, s=20, label='JAFFE')
    ax.scatter(X_ck_pca[:, 0], X_ck_pca[:, 1], 
              c='red', alpha=0.6, s=20, label='CK+')
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title('PCA - Separação entre Datasets')
    ax.legend()
    ax.grid(True, alpha=0.3)

def plot_resumo_performance(resultados, ax):
    """Plota resumo geral de performance."""
    # Coletar todas as accuracies
    all_perfs = []
    labels = []
    
    # Validação interna
    validacao = resultados['validacao_interna']
    if validacao['kfold_jaffe']:
        all_perfs.append(validacao['kfold_jaffe']['accuracy_mean'])
        labels.append('JAFFE\nIntra-dataset')
    
    if validacao['kfold_ck']:
        all_perfs.append(validacao['kfold_ck']['accuracy_mean'])
        labels.append('CK+\nIntra-dataset')
    
    # Cross-dataset
    cross_perf = resultados['cross_performance']
    if cross_perf['jaffe_to_ck'] and 'SVM' in cross_perf['jaffe_to_ck']:
        all_perfs.append(cross_perf['jaffe_to_ck']['SVM']['accuracy'])
        labels.append('JAFFE→CK+\nCross-dataset')
    
    if cross_perf['ck_to_jaffe'] and 'SVM' in cross_perf['ck_to_jaffe']:
        all_perfs.append(cross_perf['ck_to_jaffe']['SVM']['accuracy'])
        labels.append('CK+→JAFFE\nCross-dataset')
    
    if all_perfs:
        colors = ['green', 'green', 'red', 'red'][:len(all_perfs)]
        bars = ax.bar(labels, all_perfs, color=colors, alpha=0.7)
        
        ax.set_ylabel('Accuracy')
        ax.set_title('Resumo de Performance')
        ax.set_ylim(0, 1)
        
        # Adicionar valores
        for bar, perf in zip(bars, all_perfs):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{perf:.3f}', ha='center', va='bottom')
    
    ax.grid(True, alpha=0.3)

def gerar_relatorio_final_completo(resultados):
    """Gera relatório final completo."""
    
    print("\n" + "=" * 80)
    print("📊 RELATÓRIO FINAL - CROSS-DATASET DRIFT ANALYSIS")
    print("=" * 80)
    
    # Metodologia
    print("\n🔬 METODOLOGIA APLICADA:")
    print("• Datasets: JAFFE (balanceado 50/50) vs CK+ (original)")
    print("• Classes: 7 expressões (anger, disgust, fear, happy, neutral, sadness, surprise)")
    print("• Balanceamento: Data augmentation para classes minoritárias JAFFE")
    print("• Características: LBP multi-escala (radius=1,2,3; neighbors=8,16,24)")
    print("• Classificador: SVM com grid search otimizado")
    print("• Validação: Cross-dataset + LOSO + K-Fold=3")
    
    # Estatísticas de balanceamento
    stats_bal = resultados['stats_balanceamento']
    if stats_bal:
        print("\n📈 ESTATÍSTICAS DE BALANCEAMENTO:")
        total_orig = sum([s['original'] for s in stats_bal.values()])
        total_sint = sum([s['sintéticas'] for s in stats_bal.values()])
        print(f"• Amostras originais JAFFE: {total_orig}")
        print(f"• Amostras sintéticas geradas: {total_sint}")
        print(f"• Total final JAFFE: {total_orig + total_sint}")
        print("• Classes balanceadas:")
        for classe, stats in stats_bal.items():
            print(f"  - {classe}: {stats['original']} → {stats['final']} (+{stats['sintéticas']})")
    
    # Performance intra-dataset
    validacao = resultados['validacao_interna']
    print("\n📊 PERFORMANCE INTRA-DATASET:")
    
    if validacao['kfold_jaffe']:
        kf_jaffe = validacao['kfold_jaffe']
        print(f"• JAFFE K-Fold=3: {kf_jaffe['accuracy_mean']:.4f} ± {kf_jaffe['accuracy_std']:.4f}")
    
    if validacao['loso_jaffe']:
        loso_jaffe = validacao['loso_jaffe']
        print(f"• JAFFE LOSO: {loso_jaffe['accuracy_mean']:.4f} ± {loso_jaffe['accuracy_std']:.4f} ({loso_jaffe['n_folds']} folds)")
    
    if validacao['kfold_ck']:
        kf_ck = validacao['kfold_ck']
        print(f"• CK+ K-Fold=3: {kf_ck['accuracy_mean']:.4f} ± {kf_ck['accuracy_std']:.4f}")
    
    if validacao['loso_ck']:
        loso_ck = validacao['loso_ck']
        print(f"• CK+ LOSO: {loso_ck['accuracy_mean']:.4f} ± {loso_ck['accuracy_std']:.4f} ({loso_ck['n_folds']} folds)")
    
    # Performance cross-dataset
    cross_perf = resultados['cross_performance']
    print("\n🎯 PERFORMANCE CROSS-DATASET:")
    
    if cross_perf['jaffe_to_ck'] and 'SVM' in cross_perf['jaffe_to_ck']:
        acc_j2c = cross_perf['jaffe_to_ck']['SVM']['accuracy']
        f1_j2c = cross_perf['jaffe_to_ck']['SVM']['f1_macro']
        print(f"• JAFFE → CK+: {acc_j2c:.4f} accuracy, {f1_j2c:.4f} f1-macro")
    
    if cross_perf['ck_to_jaffe'] and 'SVM' in cross_perf['ck_to_jaffe']:
        acc_c2j = cross_perf['ck_to_jaffe']['SVM']['accuracy']
        f1_c2j = cross_perf['ck_to_jaffe']['SVM']['f1_macro']
        print(f"• CK+ → JAFFE: {acc_c2j:.4f} accuracy, {f1_c2j:.4f} f1-macro")
    
    # Drift detectado
    drift = resultados['drift_metrics']
    print("\n🔍 DRIFT DETECTADO:")
    if drift:
        ks_drift = drift['ks_test']['drift_percentage']
        wasserstein = drift['wasserstein']['mean_distance']
        print(f"• Drift estatístico: {ks_drift:.1f}% das características afetadas")
        print(f"• Distância Wasserstein: {wasserstein:.4f}")
        
        if 'class_distribution' in drift:
            kl_div = drift['class_distribution']['kl_divergence']
            print(f"• Divergência KL (classes): {kl_div:.4f}")
    
    # Interpretação e impacto
    print("\n💡 INTERPRETAÇÃO E IMPACTO:")
    
    # Calcular queda de performance
    if (validacao['kfold_jaffe'] and validacao['kfold_ck'] and 
        cross_perf['jaffe_to_ck'] and cross_perf['ck_to_jaffe']):
        
        intra_avg = (validacao['kfold_jaffe']['accuracy_mean'] + 
                    validacao['kfold_ck']['accuracy_mean']) / 2
        
        cross_avg = (cross_perf['jaffe_to_ck']['SVM']['accuracy'] + 
                    cross_perf['ck_to_jaffe']['SVM']['accuracy']) / 2
        
        performance_drop = intra_avg - cross_avg
        
        print(f"• Performance intra-dataset média: {intra_avg:.4f}")
        print(f"• Performance cross-dataset média: {cross_avg:.4f}")
        print(f"• Queda de performance: {performance_drop:.4f} ({performance_drop/intra_avg:.1%})")
        
        if performance_drop > 0.2:
            print("• 🚨 DRIFT SEVERO detectado - Forte incompatibilidade entre datasets")
        elif performance_drop > 0.1:
            print("• ⚠️ DRIFT MODERADO - Adaptação de domínio necessária")
        else:
            print("• ✅ DRIFT BAIXO - Boa generalização entre datasets")
    
    # Conclusões
    print("\n🎯 CONCLUSÕES PRINCIPAIS:")
    print("• Balanceamento 50/50 efetivo sem comprometer análise de drift")
    print("• Cross-dataset drift confirmado entre JAFFE e CK+")
    print("• SVM+LBP adequado para análise em datasets pequenos")
    print("• K-Fold=3 fornece validação interna confiável")
    print("• LOSO confirma robustez para generalização de sujeitos")

In [ ]:
# ========================================
# PIPELINE PRINCIPAL
# ========================================
def pipeline_completo_cross_dataset_drift():
    """
    Pipeline completo: SVM + LBP + LOSO + K-Fold=3 + Cross-Dataset Drift
    Com balanceamento 50/50 usando data augmentation
    """
    
    print("🚀 PIPELINE COMPLETO - CROSS-DATASET DRIFT ANALYSIS")
    print("=" * 80)
    
    # ========================================
    # ETAPA 1: CARREGAMENTO DOS DADOS
    # ========================================
    
    print("\n📂 ETAPA 1: Carregamento dos Dados")
    print("-" * 50)
    
    # Caminhos dos datasets (ajustar conforme necessário)
    pasta_jaffe = '..\data\jaffe'  # ORIGINAL
    pasta_ck = '..\data\ck+'       # ORIGINAL
    
    print("Carregando JAFFE original...")
    dados_jaffe_orig = carregar_imagens_por_sujeito(pasta_jaffe)
    
    print("Carregando CK+ original...")
    dados_ck_orig = carregar_imagens_por_sujeito(pasta_ck)
    
    print(f"✅ JAFFE: {len(dados_jaffe_orig)} amostras")
    print(f"✅ CK+: {len(dados_ck_orig)} amostras")
    
    # ========================================
    # ETAPA 2: BALANCEAMENTO 50/50 JAFFE
    # ========================================
    
    print("\n⚖️ ETAPA 2: Balanceamento 50/50 do JAFFE")
    print("-" * 50)
    
    dados_jaffe_balanced, stats_balanceamento = executar_balanceamento_50_50(dados_jaffe_orig)
    
    # Filtrar CK+ para mesmas classes
    classes_alvo = CONFIG_FINAL['classes_alvo']
    dados_ck_filtered = [(img, cls, suj) for img, cls, suj in dados_ck_orig 
                         if cls in classes_alvo]
    
    print(f"✅ JAFFE balanceado: {len(dados_jaffe_balanced)} amostras")
    print(f"✅ CK+ filtrado: {len(dados_ck_filtered)} amostras")
    
    # ========================================
    # ETAPA 3: EXTRAÇÃO DE CARACTERÍSTICAS LBP
    # ========================================
    
    print("\n🔍 ETAPA 3: Extração de Características LBP Multi-escala")
    print("-" * 50)
    
    print("Processando JAFFE balanceado...")
    X_jaffe, y_jaffe, subjects_jaffe = processar_dataset_lbp(
        dados_jaffe_balanced, usar_multiescala=True, mostrar_progresso=True
    )
    
    print("Processando CK+ filtrado...")
    X_ck, y_ck, subjects_ck = processar_dataset_lbp(
        dados_ck_filtered, usar_multiescala=True, mostrar_progresso=True
    )
    
    print(f"✅ JAFFE: {X_jaffe.shape}")
    print(f"✅ CK+: {X_ck.shape}")
    
    # ========================================
    # ETAPA 4: ANÁLISE DE CROSS-DATASET DRIFT
    # ========================================
    
    print("\n🔬 ETAPA 4: Análise de Cross-Dataset Drift")
    print("-" * 50)
    
    # Inicializar analisador
    analyzer = CrossDatasetDriftAnalyzer(random_state=CONFIG_FINAL['random_state'])
    
    # Adicionar datasets
    analyzer.adicionar_dataset('JAFFE', X_jaffe, y_jaffe, subjects_jaffe)
    analyzer.adicionar_dataset('CK+', X_ck, y_ck, subjects_ck)
    
    # Drift estatístico
    print("\n🔍 Calculando drift estatístico...")
    drift_metrics = analyzer.calcular_drift_estatistico('JAFFE', 'CK+')
    
    # Performance cross-dataset
    print("\n🎯 Avaliando performance cross-dataset...")
    cross_jaffe_to_ck = analyzer.avaliar_cross_performance('JAFFE', 'CK+')
    cross_ck_to_jaffe = analyzer.avaliar_cross_performance('CK+', 'JAFFE')
    
    # ========================================
    # ETAPA 5: VALIDAÇÃO INTERNA (LOSO + K-FOLD)
    # ========================================
    
    print("\n📊 ETAPA 5: Validação Interna")
    print("-" * 50)
    
    # LOSO interno
    print("\n🎯 Executando LOSO interno...")
    loso_jaffe = analyzer.executar_loso_interno('JAFFE', verbose=True)
    loso_ck = analyzer.executar_loso_interno('CK+', verbose=True)
    
    # K-Fold interno
    print(f"\n📊 Executando K-Fold={CONFIG_FINAL['k_fold_cv']} interno...")
    kfold_jaffe = analyzer.executar_kfold_interno('JAFFE', k=CONFIG_FINAL['k_fold_cv'])
    kfold_ck = analyzer.executar_kfold_interno('CK+', k=CONFIG_FINAL['k_fold_cv'])
    
    # ========================================
    # ETAPA 6: ANÁLISE E VISUALIZAÇÃO
    # ========================================
    
    print("\n📈 ETAPA 6: Análise e Visualização")
    print("-" * 50)
    
    resultados_completos = {
        'dados_processados': {
            'jaffe': (X_jaffe, y_jaffe, subjects_jaffe),
            'ck': (X_ck, y_ck, subjects_ck)
        },
        'stats_balanceamento': stats_balanceamento,
        'drift_metrics': drift_metrics,
        'cross_performance': {
            'jaffe_to_ck': cross_jaffe_to_ck,
            'ck_to_jaffe': cross_ck_to_jaffe
        },
        'validacao_interna': {
            'loso_jaffe': loso_jaffe,
            'loso_ck': loso_ck,
            'kfold_jaffe': kfold_jaffe,
            'kfold_ck': kfold_ck
        },
        'analyzer': analyzer
    }
    
    # Gerar visualizações
    gerar_visualizacoes_completas(resultados_completos)
    
    # Relatório final
    gerar_relatorio_final_completo(resultados_completos)
    
    return resultados_completos

# ========================================
# FUNÇÃO PRINCIPAL DE EXECUÇÃO
# ========================================

def executar_analise_completa():
    """
    Função principal para executar toda a análise.
    """
    print("🎯 INICIANDO ANÁLISE COMPLETA DE CROSS-DATASET DRIFT")
    print("🔧 Configurações:", CONFIG_FINAL)
    print()
    
    try:
        resultados = pipeline_completo_cross_dataset_drift()
        
        print("\n🎉 ANÁLISE CONCLUÍDA COM SUCESSO!")
        print("📊 Todos os resultados foram processados e visualizados.")
        print("📋 Relatório final gerado.")
        
        return resultados
        
    except Exception as e:
        print(f"\n❌ ERRO na análise: {e}")
        import traceback
        traceback.print_exc()
        return None

# ========================================
# VALIDAÇÃO E TESTES
# ========================================

def validar_pipeline():
    """Valida se o pipeline está funcionando corretamente."""
    print("🧪 VALIDANDO PIPELINE...")
    
    # Teste com dados sintéticos pequenos
    print("Gerando dados sintéticos para teste...")
    
    dados_teste = []
    classes = CONFIG_FINAL['classes_alvo'][:3]  # Apenas 3 classes para teste
    
    for i, classe in enumerate(classes):
        for j in range(10):  # 10 amostras por classe
            img_fake = np.random.rand(96, 96).astype(np.float32)
            sujeito_fake = f"S{i:03d}_{j}"
            dados_teste.append((img_fake, classe, sujeito_fake))
    
    print(f"✅ Dados sintéticos: {len(dados_teste)} amostras")
    
    # Testar balanceamento
    try:
        dados_bal, stats = executar_balanceamento_50_50(dados_teste)
        print(f"✅ Balanceamento: {len(dados_bal)} amostras")
    except Exception as e:
        print(f"❌ Erro no balanceamento: {e}")
        return False
    
    # Testar LBP
    try:
        X, y, subjects = processar_dataset_lbp(dados_bal[:20], mostrar_progresso=False)
        print(f"✅ LBP: {X.shape}")
    except Exception as e:
        print(f"❌ Erro no LBP: {e}")
        return False
    
    # Testar SVM
    try:
        svm = SVMClassifier()
        svm.treinar(X, y, tipo_grid='rapido', verbose=False)
        pred = svm.predizer(X[:5])
        print(f"✅ SVM: {len(pred)} predições")
    except Exception as e:
        print(f"❌ Erro no SVM: {e}")
        return False
    
    print("🎉 Validação do pipeline concluída com sucesso!")
    return True

print("✅ PIPELINE COMPLETO IMPLEMENTADO!")
print()
print("🚀 Para executar a análise completa:")
print(">>> resultados = executar_analise_completa()")
print()
print("🧪 Para validar o pipeline primeiro:")
print(">>> validar_pipeline()")
print()
print("⚙️ Configurações atuais:")
for key, value in CONFIG_FINAL.items():
    print(f"   {key}: {value}")

    

✅ PIPELINE COMPLETO IMPLEMENTADO!

🚀 Para executar a análise completa:
>>> resultados = executar_analise_completa()

🧪 Para validar o pipeline primeiro:
>>> validar_pipeline()

⚙️ Configurações atuais:
   image_size: (96, 96)
   lbp_radius: [1, 2, 3]
   lbp_neighbors: [8, 16, 24]
   lbp_method: uniform
   grid_size: (3, 3)
   random_state: 42
   k_fold_cv: 3
   classes_alvo: ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sadness', 'surprise']
   balanceamento_ratio: 1.0


In [ ]:
# ========================================
# VALIDAÇÃO DO PIPELINE
# ========================================

print("🧪 Executando validação do pipeline...")
resultado_validacao = validar_pipeline()

if resultado_validacao:
    print("🎉 Pipeline validado com sucesso!")
    print("✅ Pronto para executar análise completa!")
else:
    print("❌ Erro na validação - verifique as dependências")

🧪 Executando validação do pipeline...
🧪 VALIDANDO PIPELINE...
Gerando dados sintéticos para teste...
✅ Dados sintéticos: 30 amostras
❌ Erro no balanceamento: name 'executar_balanceamento_50_50' is not defined
❌ Erro na validação - verifique as dependências


In [ ]:
# ========================================
# CONFIGURAÇÃO DOS CAMINHOS DOS DATASETS
# ========================================

# ⚠️ AJUSTE OS CAMINHOS CONFORME SUA ESTRUTURA
pasta_jaffe = '../data/jaffe'
pasta_ck = '../data/ck+'       

# Verificar se os diretórios existem
if os.path.exists(pasta_jaffe):
    print(f"✅ JAFFE encontrado: {pasta_jaffe}")
    print(f"   Classes: {os.listdir(pasta_jaffe)}")
else:
    print(f"❌ JAFFE não encontrado: {pasta_jaffe}")

if os.path.exists(pasta_ck):
    print(f"✅ CK+ encontrado: {pasta_ck}")
    print(f"   Classes: {os.listdir(pasta_ck)}")
else:
    print(f"❌ CK+ não encontrado: {pasta_ck}")

✅ JAFFE encontrado: ../data/jaffe
   Classes: ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sadness', 'surprise']
✅ CK+ encontrado: ../data/ck+
   Classes: ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sadness', 'surprise']


In [ ]:
# ========================================
# EXECUÇÃO DA ANÁLISE COMPLETA
# ========================================

print("🚀 INICIANDO ANÁLISE COMPLETA DE CROSS-DATASET DRIFT")
print("=" * 80)

# Executar análise
resultados = executar_analise_completa()

if resultados:
    print("\n🎉 ANÁLISE CONCLUÍDA COM SUCESSO!")
    print("📊 Resultados disponíveis na variável 'resultados'")
else:
    print("\n❌ Erro na execução da análise")

🚀 INICIANDO ANÁLISE COMPLETA DE CROSS-DATASET DRIFT
🎯 INICIANDO ANÁLISE COMPLETA DE CROSS-DATASET DRIFT
🔧 Configurações: {'image_size': (96, 96), 'lbp_radius': [1, 2, 3], 'lbp_neighbors': [8, 16, 24], 'lbp_method': 'uniform', 'grid_size': (3, 3), 'random_state': 42, 'k_fold_cv': 3, 'classes_alvo': ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sadness', 'surprise'], 'balanceamento_ratio': 1.0}

🚀 PIPELINE COMPLETO - CROSS-DATASET DRIFT ANALYSIS

📂 ETAPA 1: Carregamento dos Dados
--------------------------------------------------
Carregando JAFFE original...

❌ ERRO na análise: name 'carregar_imagens_por_sujeito' is not defined

❌ Erro na execução da análise


Traceback (most recent call last):
  File "C:\Program Files\R\R-4.3.2\ipykernel_12244\3192325948.py", line 155, in executar_analise_completa
    resultados = pipeline_completo_cross_dataset_drift()
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\R\R-4.3.2\ipykernel_12244\3192325948.py", line 22, in pipeline_completo_cross_dataset_drift
    dados_jaffe_orig = carregar_imagens_por_sujeito(pasta_jaffe)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'carregar_imagens_por_sujeito' is not defined


In [ ]:
# ========================================
# ANÁLISE DETALHADA DOS RESULTADOS
# ========================================

if 'resultados' in locals() and resultados:
    print("📊 RESUMO DOS RESULTADOS:")
    print("-" * 40)
    
    # Estatísticas básicas
    jaffe_data = resultados['dados_processados']['jaffe']
    ck_data = resultados['dados_processados']['ck']
    
    print(f"📈 JAFFE processado: {jaffe_data[0].shape}")
    print(f"📈 CK+ processado: {ck_data[0].shape}")
    
    # Performance cross-dataset
    cross_perf = resultados['cross_performance']
    if cross_perf['jaffe_to_ck']:
        acc = cross_perf['jaffe_to_ck']['SVM']['accuracy']
        print(f"🎯 JAFFE → CK+: {acc:.4f} accuracy")
    
    if cross_perf['ck_to_jaffe']:
        acc = cross_perf['ck_to_jaffe']['SVM']['accuracy']
        print(f"🎯 CK+ → JAFFE: {acc:.4f} accuracy")
    
    # Drift metrics
    drift = resultados['drift_metrics']
    if drift:
        ks_drift = drift['ks_test']['drift_percentage']
        print(f"🔍 Drift detectado: {ks_drift:.1f}% das características")
else:
    print("❌ Resultados não disponíveis - execute a análise primeiro")

❌ Resultados não disponíveis - execute a análise primeiro


In [ ]:
# ========================================
# BALANCEADOR HÍBRIDO CROSS-DATASET COMPLETO
# ========================================

class BalanceadorHibrido:
    """
    Implementa estratégia híbrida de balanceamento cross-dataset
    com distribuição inteligente, mínima repetição de imagens base
    e redimensionamento preservando aspecto.
    """
    
    def __init__(self, random_state=42, target_size=(128, 128), usar_preprocessamento=True):
        self.random_state = random_state
        self.target_size = target_size
        self.usar_preprocessamento = usar_preprocessamento
        
        np.random.seed(random_state)
        random.seed(random_state)
        
        self.scores_qualidade = {}
        self.scores_unicidade = {}
        self.quotas_distribuidas = {}
        self.transformacoes_por_imagem = defaultdict(set)
        
        print(f"🔧 BalanceadorHibrido configurado:")
        print(f"   • Resolução alvo: {target_size}")
        print(f"   • Pré-processamento: {'Ativado' if usar_preprocessamento else 'Desativado'}")
        print(f"   • Preservação de aspecto: Ativada")
        
    def processar_imagem_entrada(self, imagem):
        """Processa imagem de entrada com redimensionamento inteligente."""
        # Redimensionar preservando aspecto
        img_resized = redimensionar_preservando_aspecto(imagem, self.target_size)
        
        # Aplicar pré-processamento se ativado
        if self.usar_preprocessamento:
            img_processed = preprocessar_para_otimizar_resolucao(img_resized)
        else:
            # Normalizar para 0-1 se necessário
            if img_resized.dtype != np.float32:
                if img_resized.max() > 1.0:
                    img_processed = img_resized.astype(np.float32) / 255.0
                else:
                    img_processed = img_resized.astype(np.float32)
            else:
                img_processed = img_resized
        
        return img_processed
        
    def executar_balanceamento_cross_dataset(self, dados_jaffe, dados_ck, classes_alvo):
        """Executa balanceamento híbrido cross-dataset com processamento inteligente."""
        
        print("🧠 EXECUTANDO BALANCEAMENTO HÍBRIDO CROSS-DATASET")
        print("=" * 70)
        print("• Análise de qualidade e unicidade das imagens")
        print("• Distribuição inteligente de quotas de repetição")
        print("• Geração de sintéticas com máxima diversidade")
        print("• Processamento inteligente de resolução")
        print("• Objetivo: Igualar JAFFE ao CK+ (expansão, não redução)")
        
        # Processar imagens de entrada
        print(f"\n🔄 Processando imagens com resolução {self.target_size}...")
        
        dados_jaffe_processados = []
        for img, cls, suj in dados_jaffe:
            img_processada = self.processar_imagem_entrada(img)
            dados_jaffe_processados.append((img_processada, cls, suj))
        
        dados_ck_processados = []
        for img, cls, suj in dados_ck:
            img_processada = self.processar_imagem_entrada(img)
            dados_ck_processados.append((img_processada, cls, suj))
        
        print(f"✅ Processamento concluído:")
        print(f"   • JAFFE: {len(dados_jaffe_processados)} imagens processadas")
        print(f"   • CK+: {len(dados_ck_processados)} imagens processadas")
        
        # Filtrar dados
        dados_jaffe_filt = [(img, cls, suj) for img, cls, suj in dados_jaffe_processados if cls in classes_alvo]
        dados_ck_filt = [(img, cls, suj) for img, cls, suj in dados_ck_processados if cls in classes_alvo]
        
        # Agrupar por classe
        dados_por_classe_jaffe = self._agrupar_por_classe(dados_jaffe_filt)
        dados_por_classe_ck = self._agrupar_por_classe(dados_ck_filt)
        
        # Executar balanceamento por classe
        dados_jaffe_result = []
        dados_ck_result = []
        stats_completos = {}
        
        print(f"\n📊 PLANO DE BALANCEAMENTO POR CLASSE:")
        print("Classe       JAFFE    CK+     Target   Sintéticas_J  Sintéticas_C")
        print("-" * 75)
        
        for classe in classes_alvo:
            dados_jaffe_classe = dados_por_classe_jaffe.get(classe, [])
            dados_ck_classe = dados_por_classe_ck.get(classe, [])
            
            if not dados_jaffe_classe or not dados_ck_classe:
                print(f"{classe:12} {len(dados_jaffe_classe):6d}   {len(dados_ck_classe):6d}   SKIP")
                continue
            
            # Target: igualar ao maior (estratégia de expansão)
            target = max(len(dados_jaffe_classe), len(dados_ck_classe))
            sinteticas_jaffe = max(0, target - len(dados_jaffe_classe))
            sinteticas_ck = max(0, target - len(dados_ck_classe))
            
            print(f"{classe:12} {len(dados_jaffe_classe):6d}   {len(dados_ck_classe):6d}   {target:6d}   {sinteticas_jaffe:11d}   {sinteticas_ck:11d}")
            
            # Processar JAFFE
            jaffe_processado = self._processar_dataset_classe(
                dados_jaffe_classe, dados_ck_classe, target, "JAFFE", classe
            )
            
            # Processar CK+
            ck_processado = self._processar_dataset_classe(
                dados_ck_classe, dados_jaffe_classe, target, "CK+", classe
            )
            
            dados_jaffe_result.extend(jaffe_processado)
            dados_ck_result.extend(ck_processado)
            
            stats_completos[classe] = {
                'jaffe_original': len(dados_jaffe_classe),
                'ck_original': len(dados_ck_classe),
                'target': target,
                'jaffe_sinteticas': sinteticas_jaffe,
                'ck_sinteticas': sinteticas_ck,
                'jaffe_final': len(jaffe_processado),
                'ck_final': len(ck_processado)
            }
        
        print("-" * 75)
        print(f"{'TOTAL':12} {len(dados_jaffe_result):6d}   {len(dados_ck_result):6d}")
        
        # Gerar relatório final
        self._gerar_relatorio_hibrido(stats_completos)
        
        return dados_jaffe_result, dados_ck_result, stats_completos
    
    def _agrupar_por_classe(self, dados):
        """Agrupa dados por classe."""
        grupos = defaultdict(list)
        for img, cls, suj in dados:
            grupos[cls].append((img, cls, suj))
        return dict(grupos)
    
    def _processar_dataset_classe(self, dados_classe, dados_referencia, target, dataset_name, classe):
        """Processa uma classe específica aplicando estratégia híbrida."""
        
        dados_expandido = dados_classe.copy()
        sinteticas_needed = max(0, target - len(dados_classe))
        
        if sinteticas_needed == 0:
            return dados_expandido
        
        # Análise híbrida para JAFFE, augmentation simples para CK+
        if dataset_name == "JAFFE" and sinteticas_needed > 0:
            print(f"\n🔍 Análise híbrida para {classe} (JAFFE)...")
            
            # Análise de qualidade
            scores_qualidade = self._avaliar_qualidade_imagens(dados_classe)
            
            # Análise de unicidade vs dataset de referência
            scores_unicidade = self._calcular_unicidade_vs_referencia(dados_classe, dados_referencia)
            
            # Distribuição inteligente de quotas
            quotas = self._distribuir_quotas_inteligente(
                dados_classe, scores_qualidade, scores_unicidade, sinteticas_needed
            )
            
            # Geração de sintéticas com máxima diversidade
            sinteticas_geradas = self._gerar_sinteticas_hibrido(dados_classe, quotas, classe)
            
        else:
            # CK+: augmentation simples
            sinteticas_geradas = []
            for i in range(sinteticas_needed):
                img_base, cls, suj_base = random.choice(dados_classe)
                tecnica = np.random.choice(['horizontal_flip', 'brightness_up', 'brightness_down', 'rotation_left'])
                img_sintetica = self._aplicar_transformacao_especifica(img_base, tecnica)
                suj_sintetico = f"{suj_base}_{dataset_name.lower()}_aug_{i:03d}"
                sinteticas_geradas.append((img_sintetica, cls, suj_sintetico))
        
        dados_expandido.extend(sinteticas_geradas)
        return dados_expandido
    
    def _avaliar_qualidade_imagens(self, dados_classe):
        """Avalia qualidade das imagens baseado em nitidez, contraste e entropia."""
        
        scores = {}
        
        for img, cls, suj in dados_classe:
            # Converter para uint8 se necessário
            if img.dtype != np.uint8:
                if img.max() <= 1.0:
                    img_uint8 = (img * 255).astype(np.uint8)
                else:
                    img_uint8 = img.astype(np.uint8)
            else:
                img_uint8 = img
            
            # Métricas de qualidade
            nitidez = cv2.Laplacian(img_uint8, cv2.CV_64F).var()
            contraste = np.std(img_uint8)
            entropia = shannon_entropy(img_uint8)
            edges = cv2.Canny(img_uint8, 50, 150)
            densidade_bordas = np.sum(edges > 0) / edges.size
            
            # Score composto (normalizado 0-10)
            score_nitidez = min(nitidez / 500, 10)
            score_contraste = min(contraste / 50, 10)
            score_entropia = min(entropia / 8, 10)
            score_bordas = densidade_bordas * 100
            
            score_final = (score_nitidez + score_contraste + score_entropia + score_bordas) / 4
            scores[suj] = min(score_final, 10.0)
        
        return scores
    
    def _calcular_unicidade_vs_referencia(self, dados_classe, dados_referencia):
        """Calcula unicidade das imagens comparado ao dataset de referência usando LBP."""
        
        scores_unicidade = {}
        
        # Extrair features LBP simples para comparação
        features_classe = []
        sujeitos_classe = []
        
        for img, cls, suj in dados_classe:
            features = self._extrair_features_lbp_simples(img)
            features_classe.append(features)
            sujeitos_classe.append(suj)
        
        features_referencia = []
        for img, cls, suj in dados_referencia:
            features = self._extrair_features_lbp_simples(img)
            features_referencia.append(features)
        
        features_classe = np.array(features_classe)
        features_referencia = np.array(features_referencia)
        
        # Calcular similaridade
        for i, suj in enumerate(sujeitos_classe):
            similaridades = cosine_similarity([features_classe[i]], features_referencia)[0]
            max_similaridade = np.max(similaridades)
            score_unicidade = (1 - max_similaridade) * 10
            scores_unicidade[suj] = max(score_unicidade, 0.0)
        
        return scores_unicidade
    
    def _extrair_features_lbp_simples(self, img):
        """Extrai features LBP simples para comparação."""
        if img.dtype != np.uint8:
            if img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            else:
                img = img.astype(np.uint8)
        
        lbp = local_binary_pattern(img, 8, 1, method='uniform')
        hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)
        return hist
    
    def _distribuir_quotas_inteligente(self, dados_classe, scores_qualidade, scores_unicidade, total_needed):
        """Distribui quotas de repetição baseado em qualidade e unicidade."""
        
        # Combinar scores (60% qualidade, 40% unicidade)
        scores_finais = {}
        for suj in scores_qualidade.keys():
            score_qualidade = scores_qualidade[suj]
            score_unicidade = scores_unicidade[suj]
            score_final = (score_qualidade * 0.6) + (score_unicidade * 0.4)
            scores_finais[suj] = score_final
        
        # Ordenar por score (melhores primeiro)
        sujeitos_ordenados = sorted(scores_finais.keys(), key=lambda x: scores_finais[x], reverse=True)
        
        # Distribuição baseada em tiers
        n_imagens = len(dados_classe)
        quota_base = total_needed // n_imagens
        quota_extra = total_needed % n_imagens
        
        quotas = {}
        tier_size = max(1, n_imagens // 3)
        
        for i, suj in enumerate(sujeitos_ordenados):
            if i < tier_size:  # Tier A (melhor terço)
                quota = quota_base + 2
            elif i < 2 * tier_size:  # Tier B (terço médio)
                quota = quota_base + 1
            else:  # Tier C (terço inferior)
                quota = quota_base
            
            quotas[suj] = max(quota, 0)
        
        # Distribuir quotas extras para os melhores
        extras_distribuidas = 0
        for suj in sujeitos_ordenados:
            if extras_distribuidas < quota_extra:
                quotas[suj] += 1
                extras_distribuidas += 1
            else:
                break
        
        # Ajuste fino para atingir total exato
        total_quotas = sum(quotas.values())
        if total_quotas != total_needed:
            diff = total_needed - total_quotas
            if diff > 0:
                for suj in sujeitos_ordenados[:diff]:
                    quotas[suj] += 1
            else:
                for suj in reversed(sujeitos_ordenados[:abs(diff)]):
                    quotas[suj] = max(0, quotas[suj] - 1)
        
        return quotas
    
    def _gerar_sinteticas_hibrido(self, dados_classe, quotas, classe):
        """Gera sintéticas com máxima diversidade usando quotas inteligentes."""
        
        sinteticas = []
        
        # Criar mapeamento sujeito -> dados
        mapa_sujeito = {}
        for img, cls, suj in dados_classe:
            mapa_sujeito[suj] = (img, cls, suj)
        
        # Gerar sintéticas por quota
        for suj, quota in quotas.items():
            if quota <= 0:
                continue
                
            img_base, cls, _ = mapa_sujeito[suj]
            
            # Gerar transformações únicas para esta imagem
            sinteticas_geradas = self._gerar_transformacoes_unicas(
                img_base, cls, suj, quota
            )
            
            sinteticas.extend(sinteticas_geradas)
        
        return sinteticas
    
    def _gerar_transformacoes_unicas(self, img_base, classe, sujeito_base, num_sinteticas):
        """Gera transformações únicas garantindo máxima diversidade."""
        
        sinteticas = []
        transformacoes_usadas = set()
        
        # Definir todas as combinações possíveis
        tecnicas_basicas = [
            'horizontal_flip', 'rotation_left', 'rotation_right',
            'brightness_up', 'brightness_down', 'contrast_up', 'contrast_down',
            'translation', 'noise'
        ]
        
        for i in range(num_sinteticas):
            transformacao_unica = None
            tentativas = 0
            max_tentativas = 50
            
            while tentativas < max_tentativas:
                # Escolher técnica(s)
                if np.random.random() < 0.7:  # 70% uma técnica
                    tecnicas = [np.random.choice(tecnicas_basicas)]
                else:  # 30% duas técnicas
                    tecnicas = np.random.choice(tecnicas_basicas, 2, replace=False).tolist()
                
                # Criar hash da combinação
                combinacao_hash = self._hash_combinacao(tecnicas)
                
                # Verificar se já foi usada para esta imagem
                key_imagem = f"{sujeito_base}_{combinacao_hash}"
                if key_imagem not in transformacoes_usadas:
                    transformacoes_usadas.add(key_imagem)
                    transformacao_unica = tecnicas
                    break
                
                tentativas += 1
            
            if transformacao_unica is None:
                # Fallback: usar técnica aleatória
                transformacao_unica = [np.random.choice(tecnicas_basicas)]
            
            # Aplicar transformação
            img_sintetica = img_base.copy()
            for tecnica in transformacao_unica:
                img_sintetica = self._aplicar_transformacao_especifica(img_sintetica, tecnica)
            
            # Validar qualidade
            if self._validar_qualidade_sintetica(img_sintetica, img_base):
                sujeito_sintetico = f"{sujeito_base}_hyb_{i:03d}"
                sinteticas.append((img_sintetica, classe, sujeito_sintetico))
            else:
                # Fallback seguro
                img_sintetica = self._aplicar_transformacao_especifica(img_base, 'horizontal_flip')
                sujeito_sintetico = f"{sujeito_base}_hyb_{i:03d}_safe"
                sinteticas.append((img_sintetica, classe, sujeito_sintetico))
        
        return sinteticas
    
    def _hash_combinacao(self, tecnicas):
        """Cria hash único para combinação de técnicas."""
        tecnicas_str = "_".join(sorted(tecnicas))
        return hashlib.md5(tecnicas_str.encode()).hexdigest()[:8]
    
    def _aplicar_transformacao_especifica(self, imagem, tecnica):
        """Aplica transformação específica mantendo qualidade facial."""
        if imagem.dtype != np.float32:
            imagem = imagem.astype(np.float32)
        
        h, w = imagem.shape
        center = (w // 2, h // 2)
        
        if tecnica == 'horizontal_flip':
            return cv2.flip(imagem, 1)
        
        elif tecnica == 'rotation_left':
            angle = np.random.uniform(-8, -3)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            return cv2.warpAffine(imagem, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        
        elif tecnica == 'rotation_right':
            angle = np.random.uniform(3, 8)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            return cv2.warpAffine(imagem, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        
        elif tecnica == 'brightness_up':
            factor = np.random.uniform(1.05, 1.25)
            return np.clip(imagem * factor, 0, 1)
        
        elif tecnica == 'brightness_down':
            factor = np.random.uniform(0.75, 0.95)
            return np.clip(imagem * factor, 0, 1)
        
        elif tecnica == 'contrast_up':
            alpha = np.random.uniform(1.1, 1.3)
            return np.clip(alpha * (imagem - 0.5) + 0.5, 0, 1)
        
        elif tecnica == 'contrast_down':
            alpha = np.random.uniform(0.7, 0.9)
            return np.clip(alpha * (imagem - 0.5) + 0.5, 0, 1)
        
        elif tecnica == 'translation':
            tx = np.random.randint(-5, 6)
            ty = np.random.randint(-5, 6)
            M = np.float32([[1, 0, tx], [0, 1, ty]])
            return cv2.warpAffine(imagem, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        
        elif tecnica == 'noise':
            noise = np.random.normal(0, 0.015, imagem.shape)
            return np.clip(imagem + noise, 0, 1)
        
        else:
            return imagem  # Fallback
    
    def _validar_qualidade_sintetica(self, img_sintetica, img_original):
        """Valida se a imagem sintética mantém qualidade adequada."""
        
        # Verificações básicas
        if np.any(np.isnan(img_sintetica)) or np.any(np.isinf(img_sintetica)):
            return False
        
        if img_sintetica.std() < 0.01:  # Muito uniforme
            return False
        
        # Verificar correlação com original
        try:
            correlation = np.corrcoef(img_original.flatten(), img_sintetica.flatten())[0, 1]
            if correlation < 0.2:  # Muito diferente
                return False
        except:
            pass
        
        return True
    
    def _gerar_relatorio_hibrido(self, stats_completos):
        """Gera relatório detalhado da estratégia híbrida."""
        
        print("\n" + "=" * 70)
        print("📊 RELATÓRIO FINAL - ESTRATÉGIA HÍBRIDA")
        print("=" * 70)
        
        # Resumo geral
        total_jaffe_orig = sum([s['jaffe_original'] for s in stats_completos.values()])
        total_ck_orig = sum([s['ck_original'] for s in stats_completos.values()])
        total_jaffe_final = sum([s['jaffe_final'] for s in stats_completos.values()])
        total_ck_final = sum([s['ck_final'] for s in stats_completos.values()])
        total_sinteticas = sum([s['jaffe_sinteticas'] + s['ck_sinteticas'] for s in stats_completos.values()])
        
        print(f"\n📈 RESUMO GERAL:")
        print(f"   • JAFFE: {total_jaffe_orig} → {total_jaffe_final} amostras")
        print(f"   • CK+: {total_ck_orig} → {total_ck_final} amostras")
        print(f"   • Total sintéticas geradas: {total_sinteticas}")
        print(f"   • Ganho total: {(total_jaffe_final + total_ck_final) - (total_jaffe_orig + total_ck_orig)}")
        print(f"   • Resolução processada: {self.target_size}")
        print(f"   • Pré-processamento aplicado: {'Sim' if self.usar_preprocessamento else 'Não'}")
        
        # Ratio final
        if total_jaffe_final + total_ck_final > 0:
            ratio_final = (total_ck_final / (total_jaffe_final + total_ck_final)) * 100
            print(f"\n🎯 EFICIÊNCIA DA ESTRATÉGIA:")
            print(f"   • Ratio final: {100-ratio_final:.1f}% JAFFE / {ratio_final:.1f}% CK+")
            print(f"   • Desvio do 50/50: {abs(50-ratio_final):.1f} pontos percentuais")
            
            if abs(50 - ratio_final) < 3:
                print("   🎉 EXCELENTE: Balanceamento quase perfeito!")
            elif abs(50 - ratio_final) < 7:
                print("   ✅ BOM: Balanceamento adequado.")
            else:
                print("   ⚠️ ACEITÁVEL: Pequeno desvio do ideal.")


# ========================================
# FUNÇÃO PRINCIPAL PARA USAR NO NOTEBOOK
# ========================================

def executar_balanceamento_hibrido_com_resolucao_otimizada(dados_jaffe, dados_ck, 
                                                          classes_alvo=None, 
                                                          target_size=(128, 128), 
                                                          usar_preprocessamento=True,
                                                          random_state=42):
    """
    Função principal para executar balanceamento híbrido com redimensionamento inteligente.
    
    Args:
        dados_jaffe: Lista de tuplas (imagem, classe, sujeito) do JAFFE
        dados_ck: Lista de tuplas (imagem, classe, sujeito) do CK+
        classes_alvo: Lista de classes para processar (None = todas disponíveis)
        target_size: Resolução alvo (width, height)
        usar_preprocessamento: Se True, aplica técnicas de otimização
        random_state: Seed para reprodutibilidade
    
    Returns:
        dados_jaffe_balanced: JAFFE balanceado
        dados_ck_balanced: CK+ balanceado
        stats_balanceamento: Estatísticas detalhadas
    """
    
    # Definir classes padrão se não especificadas
    if classes_alvo is None:
        classes_jaffe = set([cls for _, cls, _ in dados_jaffe])
        classes_ck = set([cls for _, cls, _ in dados_ck])
        classes_alvo = sorted(list(classes_jaffe.intersection(classes_ck)))
        print(f"🎯 Classes detectadas automaticamente: {classes_alvo}")
    
    # Inicializar balanceador
    balanceador = BalanceadorHibrido(
        random_state=random_state,
        target_size=target_size,
        usar_preprocessamento=usar_preprocessamento
    )
    
    print(f"\n🚀 Iniciando processamento com configurações:")
    print(f"   • Classes alvo: {len(classes_alvo)} classes")
    print(f"   • Resolução: {target_size}")
    print(f"   • Pré-processamento: {'Ativado' if usar_preprocessamento else 'Desativado'}")
    
    # Executar balanceamento
    dados_jaffe_balanced, dados_ck_balanced, stats = balanceador.executar_balanceamento_cross_dataset(
        dados_jaffe, dados_ck, classes_alvo
    )
    
    print(f"\n🎉 BALANCEAMENTO HÍBRIDO CONCLUÍDO!")
    print(f"   • JAFFE balanceado: {len(dados_jaffe_balanced)} amostras")
    print(f"   • CK+ processado: {len(dados_ck_balanced)} amostras")
    print(f"   • Resolução final: {target_size}")
    print(f"   • Preservação de aspecto: Aplicada")
    
    return dados_jaffe_balanced, dados_ck_balanced, stats


# ========================================
# FUNÇÕES AUXILIARES
# ========================================

def gerar_relatorio_detalhado(stats_balanceamento):
    """Gera relatório detalhado das estatísticas de balanceamento."""
    print("\n📊 RELATÓRIO DETALHADO DE BALANCEAMENTO")
    print("=" * 60)
    
    for classe, stats in stats_balanceamento.items():
        print(f"\n🎭 CLASSE: {classe.upper()}")
        print(f"   📈 Original JAFFE: {stats['jaffe_original']}")
        print(f"   📈 Original CK+: {stats['ck_original']}")
        print(f"   🎯 Target: {stats['target']}")
        print(f"   🔧 Sintéticas JAFFE: {stats['jaffe_sinteticas']}")
        print(f"   🔧 Sintéticas CK+: {stats['ck_sinteticas']}")
        print(f"   ✅ Final JAFFE: {stats['jaffe_final']}")
        print(f"   ✅ Final CK+: {stats['ck_final']}")
        
        # Calcular eficiência
        if stats['jaffe_original'] > 0:
            aumento_jaffe = ((stats['jaffe_final'] - stats['jaffe_original']) / stats['jaffe_original']) * 100
            print(f"   📊 Aumento JAFFE: +{aumento_jaffe:.1f}%")

def verificar_qualidade_imagens(dados, num_amostras=10):
    """Verifica a qualidade geral das imagens processadas."""
    print(f"🔍 ANÁLISE DE QUALIDADE - {num_amostras} amostras")
    print("-" * 40)
    
    indices = np.random.choice(len(dados), min(num_amostras, len(dados)), replace=False)
    
    nitidez_scores = []
    contraste_scores = []
    entropia_scores = []
    
    for idx in indices:
        img, cls, suj = dados[idx]
        
        # Converter para uint8 se necessário
        if img.dtype != np.uint8:
            if img.max() <= 1.0:
                img_uint8 = (img * 255).astype(np.uint8)
            else:
                img_uint8 = img.astype(np.uint8)
        else:
            img_uint8 = img
        
        # Calcular métricas
        nitidez = cv2.Laplacian(img_uint8, cv2.CV_64F).var()
        contraste = np.std(img_uint8)
        entropia = shannon_entropy(img_uint8)
        
        nitidez_scores.append(nitidez)
        contraste_scores.append(contraste)
        entropia_scores.append(entropia)
    
    stats = {
        'nitidez_media': np.mean(nitidez_scores),
        'contraste_medio': np.mean(contraste_scores),
        'entropia_media': np.mean(entropia_scores),
        'nitidez_std': np.std(nitidez_scores),
        'contraste_std': np.std(contraste_scores),
        'entropia_std': np.std(entropia_scores)
    }
    
    print(f"📊 Nitidez: {stats['nitidez_media']:.1f} ± {stats['nitidez_std']:.1f}")
    print(f"📊 Contraste: {stats['contraste_medio']:.1f} ± {stats['contraste_std']:.1f}")
    print(f"📊 Entropia: {stats['entropia_media']:.2f} ± {stats['entropia_std']:.2f}")
    
    return stats

print("✅ BALANCEADOR HÍBRIDO IMPLEMENTADO!")
print("🎯 Principais recursos:")
print("   • Análise de qualidade e unicidade")
print("   • Distribuição inteligente de quotas")
print("   • 9 transformações diferentes")
print("   • Preservação de aspecto")
print("   • Pré-processamento para baixa resolução")
print("   • Validação de qualidade integrada")
print("\n🚀 Use: executar_balanceamento_hibrido_com_resolucao_otimizada(dados_jaffe, dados_ck)")